<a href="https://colab.research.google.com/github/cdascientist/Vis/blob/master/vis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title 1: INFRASTRUCTURE — DynamicSection · DynamicProbe · JITObject · TandemCalc · HF · HFEngine · PipelineTelemetry · StoryLog · VarTracker · ScanEventTemporalMatcher · JITPipelineArchitecture

from __future__ import annotations
import math
import datetime, os, logging, time, re, threading
from dataclasses import dataclass, field
from typing import Optional, Any


# ── DynamicSection ────────────────────────────────────────────────────────────

class DynamicSection:
    """Generic attribute bag.  __getattr__ returns None, never raises."""
    def __init__(self): self._data = {}
    def __getattr__(self, k): return self._data.get(k)
    def __setattr__(self, k, v):
        if k == '_data': super().__setattr__(k, v)
        else: self._data[k] = v
    def to_dict(self):
        return {k: (v.to_dict() if isinstance(v, DynamicSection) else v)
                for k, v in self._data.items()}


# ── DynamicProbe ──────────────────────────────────────────────────────────────

@dataclass
class DynamicProbe:
    uid:       str            = ""
    payload:   DynamicSection = field(default_factory=DynamicSection)
    telemetry: Optional[Any]  = None
    def __post_init__(self):
        self.payload.stat_1 = 1; self.payload.stat_2 = 2; self.payload.stat_3 = 3
        self.payload.stat_4 = 4; self.payload.stat_5 = 5


# ── JITObject ─────────────────────────────────────────────────────────────────

@dataclass
class JITObject(DynamicProbe):
    jit_metrics:      DynamicSection = field(default_factory=DynamicSection)
    tandem_interface: DynamicSection = field(default_factory=DynamicSection)   # THFPI SHARED BUS — never rename

    def __post_init__(self):
        super().__post_init__()
        if not self.uid: self.uid = "JIT-CORE-ENGINE"

        self.jit_metrics.embedded_results = DynamicSection()
        self.tandem_interface.results     = DynamicSection()

        # ── THFPI CENTROID STORE ──────────────────────────────────────────────
        # Initialised empty here so all three centroid fields are addressable
        # by dot-path before any pipeline fires.
        # Populated atomically by FactoryOrchestrator after ACTIVATION POINT 3/3
        # from the TC-PIPE result on tandem_interface.thfpi_group1_consolidated.
        # Fields written:
        #   .centroid_0_primary          float — real primary probe centroid
        #   .centroid_1_synthetic        float — synthetic midpoint (NOVEL)
        #   .centroid_2_secondary        float — real secondary probe centroid
        #   .kmeans_synthetic_seed       float — initial seed value for c1
        #   .kmeans_iteration_count      int   — always 0 for symmetric pair
        #   .norm_primary                float — day-fraction of primary ts
        #   .norm_secondary              float — day-fraction of secondary ts
        #   .norm_anchor_utc_string      str   — midnight UTC anchor
        #   .group_id                    str   — which scan_event pair
        self.tandem_interface.kmeans_centroids = DynamicSection()

        # ── THFPI VELOCITY SUB-COMPONENT ─────────────────────────────────────
        # Initialised empty here so all velocity fields are addressable by
        # dot-path before any pipeline fires.
        # Populated atomically by FactoryOrchestrator after ACTIVATION POINT 3/3
        # from the TC-PIPE Stage 4 (TC-V) result on thfpi_group1_consolidated.
        # Fields written:
        #   .group_id                        str
        #   .synthetic_centroid_position     float — same as kmeans_centroids.centroid_1_synthetic
        #   .real_centroid_count             int   — N (2 now; scales to thousands)
        #   .per_centroid_signed_tensions    list  — [tension_0, tension_1, ...]
        #   .sum_of_squared_tensions         float
        #   .cumulative_magnitude            float — sqrt(Σ tension_i²)  NOVEL
        #   .velocity_in_minutes             float
        #   .tension_symmetry_indicator      float — Σ signed tensions; 0 = balanced
        #   .mean_absolute_tension           float
        #   .velocity_normalized             float — vel / mean_abs = sqrt(N) symmetric  NOVEL
        #   .velocity_normalized_sqrt_n_reference   float — sqrt(N) baseline
        #   .velocity_normalized_deviation   float — departure from sqrt(N); 0 = symmetric
        #   .conjunctive_median_delta_for_ratio float
        #   .decimal_velocity_50fig          str   — 50-figure Decimal precision string
        #   .decimal_velocity_minutes_50fig  str
        #   .decimal_symmetry_indicator_50fig str  — '0E-17' for symmetric (true zero)
        #   .decimal_normalized_50fig        str   — sqrt(2) to 50 figures symmetric
        #   .decimal_normalized_deviation_50fig str
        self.tandem_interface.synthetic_centroid_velocity_subcomponent = DynamicSection()

    def evaluate_regex(self, pattern, text):
        matches = re.findall(pattern, str(text))
        return float(matches[0]) if matches else 0.0
    def compute_addition(self, a, b):    return float(a or 0) + float(b or 0)
    def compute_subtraction(self, a, b): return float(a or 0) - float(b or 0)
    def signature(self, data):           return f"SEAL_HASH_{data}"


# ── TandemCalc — THFPI EMBEDDED FUNCTION LIBRARY ─────────────────────────────

class TandemCalc:

    # ── standard checkpoint functions ────────────────────────────────────────
    @staticmethod
    def manual_phase_calc(a, b):   return float(a or 0) * float(b or 0)        # [TC-01]
    @staticmethod
    def checkpoint_entry(a, b):    return f"ENTRY_SYNC_[{a}]_[{b}]"            # [TC-02]
    @staticmethod
    def checkpoint_between(a, b):  return f"TRANSIT_SYNC_[{a}]_[{b}]"         # [TC-03]
    @staticmethod
    def checkpoint_exit(a, b):     return f"EXIT_SYNC_[{a}]_[{b}]"            # [TC-04]
    @staticmethod
    def subtract(a, b):            return float(a or 0) - float(b or 0)        # [TC-05]

    # ── [TC-N] normalize_to_day_fraction ─────────────────────────────────────
    @staticmethod
    def normalize_to_day_fraction(ts_primary_utc_string, ts_secondary_utc_string):
        from datetime import datetime as _dt
        _fmt = "%Y-%m-%d %H:%M:%S"                                             # [TC-N1]
        _day = 86400.0                                                          # [TC-N2]
        ep   = _dt.strptime(ts_primary_utc_string,   _fmt).timestamp()         # [TC-N3]
        es   = _dt.strptime(ts_secondary_utc_string, _fmt).timestamp()         # [TC-N4]
        pri  = _dt.strptime(ts_primary_utc_string,   _fmt)                     # [TC-N5]
        anc  = pri.strftime("%Y-%m-%d") + " 00:00:00"                          # [TC-N6]
        ea   = _dt.strptime(anc, _fmt).timestamp()                             # [TC-N7]
        np_  = (ep - ea) / _day                                                # [TC-N8]
        ns   = (es - ea) / _day                                                # [TC-N9]
        return {"norm_primary": np_, "norm_secondary": ns,                     # [TC-N11]
                "signed_delta": np_ - ns, "anchor_utc_string": anc, "anchor_epoch": ea}

    # ── [TC-K] kmeans_three_centroid_synthetic ────────────────────────────────
    @staticmethod
    def kmeans_three_centroid_synthetic(norm_primary, norm_secondary):
        _max = 100                                                              # [TC-K1]
        syn  = (norm_primary + norm_secondary) / 2.0                           # [TC-K2] NOVEL synthetic seed
        c0, c1, c2 = norm_primary, syn, norm_secondary                         # [TC-K3]
        iters = 0                                                               # [TC-K4]
        for _ in range(_max):                                                   # [TC-K5]
            d0 = [v for v in [norm_primary, norm_secondary]
                  if abs(v-c0) <= abs(v-c1) and abs(v-c0) <= abs(v-c2)]
            d1 = [v for v in [norm_primary, norm_secondary]
                  if abs(v-c1) <  abs(v-c0) and abs(v-c1) <= abs(v-c2)]       # [TC-K7] always empty
            d2 = [v for v in [norm_primary, norm_secondary]
                  if abs(v-c2) <  abs(v-c0) and abs(v-c2) <  abs(v-c1)]
            n0 = sum(d0)/len(d0) if d0 else c0                                 # [TC-K8]
            n1 = sum(d1)/len(d1) if d1 else c1                                 # [TC-K9]
            n2 = sum(d2)/len(d2) if d2 else c2                                 # [TC-K10]
            if n0 == c0 and n1 == c1 and n2 == c2: break                      # [TC-K11]
            c0, c1, c2 = n0, n1, n2; iters += 1                               # [TC-K13]
        s = sorted([c0, c1, c2])                                                # [TC-K14]
        return {"centroid_0_primary": s[0], "centroid_1_synthetic_midpoint": s[1],
                "centroid_2_secondary": s[2], "synthetic_seed": syn, "iteration_count": iters}

    # ── [TC-C] conjunctive_median_delta ──────────────────────────────────────
    @staticmethod
    def conjunctive_median_delta(c0, c1_syn, c2):
        d01 = c1_syn - c0                                                       # [TC-C1]
        d12 = c2 - c1_syn                                                      # [TC-C2]
        s   = sorted([d01, d12])
        med = (s[0] + s[1]) / 2.0                                              # [TC-C3] NOVEL
        return {"diff_0_to_1": d01, "diff_1_to_2": d12,
                "conjunctive_median": med, "extrapolated_delta": med}

    # ═══════════════════════════════════════════════════════════════════════════
    # [TC-V] synthetic_centroid_tension_velocity   NOVEL
    # ═══════════════════════════════════════════════════════════════════════════
    #
    # Computes per-centroid signed tension vectors against the synthetic centroid,
    # cumulative magnitude velocity (Euclidean norm of all tensions), normalized
    # velocity, and a 50-figure Decimal recomputation of every value.
    #
    # TENSION:  tension_i = real_centroid_i − synthetic_centroid
    #   negative → centroid LEFT  of synthetic (probe arrived earlier)
    #   positive → centroid RIGHT of synthetic (probe arrived later)
    #
    # CUMULATIVE MAGNITUDE VELOCITY:  sqrt( Σ tension_i² )
    #   Euclidean norm; scales to N probes by extending the input list.
    #   For symmetric 2-probe: velocity = conjunctive_median × √2
    #
    # NORMALIZED VELOCITY:  velocity / mean_absolute_tension
    #   Dimensionless.  Equals √N for a perfectly symmetric population.
    #   Departs from √N as temporal asymmetry grows.
    #   Comparable across all probe groups regardless of absolute gap size.
    #
    # DECIMAL LAYER B — 50-figure precision:
    #   repr() is used for real centroids (exact IEEE 754 rational value).
    #   Synthetic midpoint is recomputed as Σ(real) / N in pure Decimal,
    #   avoiding the rounding error in the stored float c1.
    #   Symmetry indicator resolves to '0E-17' (exact zero) for symmetric pair.

    @staticmethod
    def synthetic_centroid_tension_velocity(
            synthetic_centroid_day_fraction_position,
            list_of_all_real_centroid_day_fraction_positions):

        from decimal import Decimal, getcontext
        getcontext().prec = 50                                                 # [TC-V-00]

        # ── Layer A: float64 ──────────────────────────────────────────────────
        _syn = synthetic_centroid_day_fraction_position                        # [TC-V-01]
        _n   = len(list_of_all_real_centroid_day_fraction_positions)           # [TC-V-02]

        per_centroid_signed_tension_vector_list = [                            # [TC-V-03]
            _c - _syn for _c in list_of_all_real_centroid_day_fraction_positions
        ]
        per_centroid_squared_tension_value_list = [                            # [TC-V-04]
            _t ** 2 for _t in per_centroid_signed_tension_vector_list
        ]
        _sum_sq = sum(per_centroid_squared_tension_value_list)                 # [TC-V-05]
        cumulative_magnitude_velocity_in_day_fractions = (                    # [TC-V-06]
            math.sqrt(_sum_sq) if _sum_sq > 0.0 else 0.0
        )
        cumulative_magnitude_velocity_in_minutes = (                          # [TC-V-07]
            cumulative_magnitude_velocity_in_day_fractions * 1440.0
        )
        tension_symmetry_indicator = sum(per_centroid_signed_tension_vector_list)  # [TC-V-08]
        mean_absolute_tension = (                                              # [TC-V-09]
            sum(abs(_t) for _t in per_centroid_signed_tension_vector_list) / _n
        ) if _n > 0 else 0.0

        # ── Normalized velocity ───────────────────────────────────────────────
        # velocity_normalized = cumulative_magnitude / mean_absolute_tension
        # = √N for symmetric population; departs from √N with asymmetry.
        velocity_normalized = (                                                # [TC-V-10]
            cumulative_magnitude_velocity_in_day_fractions / mean_absolute_tension
            if mean_absolute_tension > 0.0 else 0.0
        )
        velocity_normalized_sqrt_n_reference = (                               # [TC-V-11]
            math.sqrt(_n) if _n > 0 else 0.0
        )
        velocity_normalized_deviation_from_sqrt_n = (                         # [TC-V-12]
            velocity_normalized - velocity_normalized_sqrt_n_reference
        )

        # ── Layer B: Decimal 50-figure ────────────────────────────────────────
        # repr() gives the exact rational value of each IEEE 754 double.
        # Synthetic midpoint recomputed as Σ(real) / N in pure Decimal so
        # tensions are exact mirror images and symmetry indicator = 0E-17.
        _D_real   = [Decimal(repr(_c))                                         # [TC-V-13]
                     for _c in list_of_all_real_centroid_day_fraction_positions]
        _D_syn    = sum(_D_real) / Decimal(_n)                                 # [TC-V-14] exact midpoint in Decimal
        _D_tensions   = [_c - _D_syn for _c in _D_real]                       # [TC-V-15]
        _D_sum_sq     = sum(_t ** 2 for _t in _D_tensions)                    # [TC-V-16]
        _D_velocity   = _D_sum_sq.sqrt() if _D_sum_sq > 0 else Decimal(0)     # [TC-V-17]
        _D_vel_min    = _D_velocity * Decimal(1440)                            # [TC-V-18]
        _D_symmetry   = sum(_D_tensions)                                       # [TC-V-19] '0E-17' symmetric
        _D_mean_abs   = sum(abs(_t) for _t in _D_tensions) / Decimal(_n)      # [TC-V-20]
        _D_normalized = (                                                      # [TC-V-21]
            _D_velocity / _D_mean_abs if _D_mean_abs > 0 else Decimal(0)
        )
        _D_sqrt_n     = Decimal(_n).sqrt()                                     # [TC-V-22]
        _D_norm_dev   = _D_normalized - _D_sqrt_n                             # [TC-V-23]

        return {
            # Layer A — float64
            "synthetic_centroid_position":               _syn,                 # [TC-V-OUT-01]
            "real_centroid_count":                       _n,                   # [TC-V-OUT-02]
            "per_centroid_signed_tension_vectors":       per_centroid_signed_tension_vector_list,  # [TC-V-OUT-03]
            "per_centroid_squared_tension_values":       per_centroid_squared_tension_value_list,  # [TC-V-OUT-04]
            "sum_of_all_squared_tensions":               _sum_sq,              # [TC-V-OUT-05]
            "cumulative_magnitude_velocity":             cumulative_magnitude_velocity_in_day_fractions,  # [TC-V-OUT-06]
            "velocity_in_minutes":                       cumulative_magnitude_velocity_in_minutes,  # [TC-V-OUT-07]
            "tension_symmetry_indicator":                tension_symmetry_indicator,                # [TC-V-OUT-08]
            "mean_absolute_tension":                     mean_absolute_tension,                     # [TC-V-OUT-09]
            "velocity_normalized":                       velocity_normalized,                       # [TC-V-OUT-10] NOVEL
            "velocity_normalized_sqrt_n_reference":      velocity_normalized_sqrt_n_reference,      # [TC-V-OUT-11]
            "velocity_normalized_deviation_from_sqrt_n": velocity_normalized_deviation_from_sqrt_n, # [TC-V-OUT-12]
            # Layer B — Decimal 50-figure strings
            "decimal_velocity_50fig":                    str(_D_velocity),     # [TC-V-OUT-13]
            "decimal_velocity_minutes_50fig":            str(_D_vel_min),      # [TC-V-OUT-14]
            "decimal_symmetry_indicator_50fig":          str(_D_symmetry),     # [TC-V-OUT-15] '0E-17' symmetric
            "decimal_normalized_50fig":                  str(_D_normalized),   # [TC-V-OUT-16]
            "decimal_normalized_deviation_50fig":        str(_D_norm_dev),     # [TC-V-OUT-17]
        }

    # ── composable chain groups 1-4 (TC-P1 … TC-P4) ─────────────────────────

    @staticmethod
    def _pipeline_body(ts_pri, ts_sec, tag):
        norm   = TandemCalc.normalize_to_day_fraction(ts_pri, ts_sec)
        kmeans = TandemCalc.kmeans_three_centroid_synthetic(
                     norm["norm_primary"], norm["norm_secondary"])
        delta  = TandemCalc.conjunctive_median_delta(
                     kmeans["centroid_0_primary"],
                     kmeans["centroid_1_synthetic_midpoint"],
                     kmeans["centroid_2_secondary"])
        return {
            "norm_primary":                         norm["norm_primary"],
            "norm_secondary":                       norm["norm_secondary"],
            "norm_signed_delta":                    norm["signed_delta"],
            "norm_anchor_utc_string":               norm["anchor_utc_string"],
            "norm_anchor_epoch":                    norm["anchor_epoch"],
            "kmeans_centroid_0_primary":            kmeans["centroid_0_primary"],
            "kmeans_centroid_1_synthetic_midpoint": kmeans["centroid_1_synthetic_midpoint"],
            "kmeans_centroid_2_secondary":          kmeans["centroid_2_secondary"],
            "kmeans_synthetic_seed":                kmeans["synthetic_seed"],
            "kmeans_iteration_count":               kmeans["iteration_count"],
            "delta_diff_0_to_1":                    delta["diff_0_to_1"],
            "delta_diff_1_to_2":                    delta["diff_1_to_2"],
            "delta_conjunctive_median":             delta["conjunctive_median"],
            "delta_extrapolated":                   delta["extrapolated_delta"],
        }

    @staticmethod
    def thfpi_group1_pipeline(a, b): return TandemCalc._pipeline_body(a, b, "P1")  # [TC-P1]
    @staticmethod
    def thfpi_group2_pipeline(a, b): return TandemCalc._pipeline_body(a, b, "P2")  # [TC-P2]
    @staticmethod
    def thfpi_group3_pipeline(a, b): return TandemCalc._pipeline_body(a, b, "P3")  # [TC-P3]
    @staticmethod
    def thfpi_group4_pipeline(a, b): return TandemCalc._pipeline_body(a, b, "P4")  # [TC-P4]

    # ═══════════════════════════════════════════════════════════════════════════
    # [TC-PIPE] CONSOLIDATED FOUR-STAGE PIPELINE   NOVEL
    # ═══════════════════════════════════════════════════════════════════════════
    #
    # Stage 1  [TC-PIPE-S1]  24-hour calendar-day normalization        (TC-N)
    # Stage 2  [TC-PIPE-S2]  k-means k=3 with synthetic midpoint seed  (TC-K)
    # Stage 3  [TC-PIPE-S3]  conjunctive median of adjacent half-gaps  (TC-C)
    # Stage 4  [TC-PIPE-S4]  synthetic centroid tension velocity        (TC-V)
    #
    # Return dict has 31 keys: OUT-01..14 (Stages 1-3) + OUT-15..31 (Stage 4).
    # All Stage 4 keys are prefixed "velocity_" for unambiguous identification.
    # Centroid keys (kmeans_centroid_0_primary, _1_synthetic_midpoint,
    # _2_secondary) are in OUT-06..08 and are written to
    # tandem_interface.kmeans_centroids by FactoryOrchestrator.

    @staticmethod
    def thfpi_probe_pair_utc_timestamps_to_conjunctive_median_delta_via_day_normalized_synthetic_midpoint_seeded_kmeans_with_synthetic_centroid_tension_velocity(
            primary_probe_scan_event_utc_timestamp_string,
            secondary_probe_scan_event_utc_timestamp_string):

        # ── [TC-PIPE-S1] NORMALIZATION ────────────────────────────────────────
        from datetime import datetime as _dt                                   # [TC-PIPE-S1-01]
        _fmt = "%Y-%m-%d %H:%M:%S"                                            # [TC-PIPE-S1-02]
        _day = 86400.0                                                         # [TC-PIPE-S1-03]
        _ep  = _dt.strptime(primary_probe_scan_event_utc_timestamp_string,   _fmt).timestamp()   # [TC-PIPE-S1-04]
        _es  = _dt.strptime(secondary_probe_scan_event_utc_timestamp_string, _fmt).timestamp()   # [TC-PIPE-S1-05]
        _pri = _dt.strptime(primary_probe_scan_event_utc_timestamp_string,   _fmt)               # [TC-PIPE-S1-06]
        _anc_str = _pri.strftime("%Y-%m-%d") + " 00:00:00"                   # [TC-PIPE-S1-07]
        _ea  = _dt.strptime(_anc_str, _fmt).timestamp()                       # [TC-PIPE-S1-08]
        stage_1_norm_primary   = (_ep - _ea) / _day                           # [TC-PIPE-S1-09]
        stage_1_norm_secondary = (_es - _ea) / _day                           # [TC-PIPE-S1-10]
        stage_1_signed_delta   = stage_1_norm_primary - stage_1_norm_secondary  # [TC-PIPE-S1-11]

        # ── [TC-PIPE-S2] K-MEANS k=3 SYNTHETIC SEED ──────────────────────────
        _max = 100                                                             # [TC-PIPE-S2-01]
        stage_2_synthetic_seed = (stage_1_norm_primary + stage_1_norm_secondary) / 2.0  # [TC-PIPE-S2-02]
        _c0, _c1, _c2 = stage_1_norm_primary, stage_2_synthetic_seed, stage_1_norm_secondary  # [TC-PIPE-S2-03]
        _iters = 0                                                             # [TC-PIPE-S2-04]
        for _ in range(_max):                                                  # [TC-PIPE-S2-05]
            _d0 = [v for v in [stage_1_norm_primary, stage_1_norm_secondary]
                   if abs(v-_c0) <= abs(v-_c1) and abs(v-_c0) <= abs(v-_c2)]
            _d1 = [v for v in [stage_1_norm_primary, stage_1_norm_secondary]
                   if abs(v-_c1) <  abs(v-_c0) and abs(v-_c1) <= abs(v-_c2)]
            _d2 = [v for v in [stage_1_norm_primary, stage_1_norm_secondary]
                   if abs(v-_c2) <  abs(v-_c0) and abs(v-_c2) <  abs(v-_c1)]
            _n0 = sum(_d0)/len(_d0) if _d0 else _c0
            _n1 = sum(_d1)/len(_d1) if _d1 else _c1
            _n2 = sum(_d2)/len(_d2) if _d2 else _c2
            if _n0 == _c0 and _n1 == _c1 and _n2 == _c2: break               # [TC-PIPE-S2-06]
            _c0, _c1, _c2 = _n0, _n1, _n2; _iters += 1                      # [TC-PIPE-S2-07]
        _sorted = sorted([_c0, _c1, _c2])                                     # [TC-PIPE-S2-08]
        stage_2_c0 = _sorted[0]                                               # [TC-PIPE-S2-09] real primary
        stage_2_c1 = _sorted[1]                                               # [TC-PIPE-S2-10] synthetic midpoint — never moves
        stage_2_c2 = _sorted[2]                                               # [TC-PIPE-S2-11] real secondary
        stage_2_iteration_count = _iters                                      # [TC-PIPE-S2-12]

        # ── [TC-PIPE-S3] CONJUNCTIVE MEDIAN ───────────────────────────────────
        stage_3_diff_0_to_1 = stage_2_c1 - stage_2_c0                        # [TC-PIPE-S3-01]
        stage_3_diff_1_to_2 = stage_2_c2 - stage_2_c1                        # [TC-PIPE-S3-02]
        _s3 = sorted([stage_3_diff_0_to_1, stage_3_diff_1_to_2])
        stage_3_conjunctive_median = (_s3[0] + _s3[1]) / 2.0                 # [TC-PIPE-S3-03] NOVEL

        # ── [TC-PIPE-S4] SYNTHETIC CENTROID TENSION VELOCITY ─────────────────
        _tc_v_result = TandemCalc.synthetic_centroid_tension_velocity(         # [TC-PIPE-S4-02]
            stage_2_c1,
            [stage_2_c0, stage_2_c2]                                          # [TC-PIPE-S4-01] list grows to N probes
        )
        stage_4_synthetic_position       = _tc_v_result["synthetic_centroid_position"]              # [TC-PIPE-S4-03]
        stage_4_real_centroid_count      = _tc_v_result["real_centroid_count"]                      # [TC-PIPE-S4-04]
        stage_4_signed_tensions          = _tc_v_result["per_centroid_signed_tension_vectors"]      # [TC-PIPE-S4-05]
        stage_4_squared_tensions         = _tc_v_result["per_centroid_squared_tension_values"]      # [TC-PIPE-S4-06]
        stage_4_sum_squared              = _tc_v_result["sum_of_all_squared_tensions"]              # [TC-PIPE-S4-07]
        stage_4_velocity_day_frac        = _tc_v_result["cumulative_magnitude_velocity"]            # [TC-PIPE-S4-08]
        stage_4_velocity_minutes         = _tc_v_result["velocity_in_minutes"]                      # [TC-PIPE-S4-09]
        stage_4_symmetry_indicator       = _tc_v_result["tension_symmetry_indicator"]               # [TC-PIPE-S4-10]
        stage_4_mean_abs_tension         = _tc_v_result["mean_absolute_tension"]                    # [TC-PIPE-S4-11]
        stage_4_velocity_normalized      = _tc_v_result["velocity_normalized"]                      # [TC-PIPE-S4-12] NOVEL
        stage_4_velocity_norm_sqrt_n_ref = _tc_v_result["velocity_normalized_sqrt_n_reference"]     # [TC-PIPE-S4-13]
        stage_4_velocity_norm_deviation  = _tc_v_result["velocity_normalized_deviation_from_sqrt_n"] # [TC-PIPE-S4-14]
        stage_4_decimal_velocity_50fig   = _tc_v_result["decimal_velocity_50fig"]                   # [TC-PIPE-S4-15]
        stage_4_decimal_vel_min_50fig    = _tc_v_result["decimal_velocity_minutes_50fig"]           # [TC-PIPE-S4-16]
        stage_4_decimal_symmetry_50fig   = _tc_v_result["decimal_symmetry_indicator_50fig"]         # [TC-PIPE-S4-17]
        stage_4_decimal_normalized_50fig = _tc_v_result["decimal_normalized_50fig"]                 # [TC-PIPE-S4-18]
        stage_4_decimal_norm_dev_50fig   = _tc_v_result["decimal_normalized_deviation_50fig"]       # [TC-PIPE-S4-19]

        return {
            # Stage 1 — normalization
            "norm_primary":                               stage_1_norm_primary,           # [TC-PIPE-OUT-01]
            "norm_secondary":                             stage_1_norm_secondary,         # [TC-PIPE-OUT-02]
            "norm_signed_delta":                          stage_1_signed_delta,           # [TC-PIPE-OUT-03]
            "norm_anchor_utc_string":                     _anc_str,                       # [TC-PIPE-OUT-04]
            "norm_anchor_epoch":                          _ea,                            # [TC-PIPE-OUT-05]
            # Stage 2 — k-means centroids  ← written to tandem_interface.kmeans_centroids
            "kmeans_centroid_0_primary":                  stage_2_c0,                     # [TC-PIPE-OUT-06]
            "kmeans_centroid_1_synthetic_midpoint":       stage_2_c1,                     # [TC-PIPE-OUT-07]
            "kmeans_centroid_2_secondary":                stage_2_c2,                     # [TC-PIPE-OUT-08]
            "kmeans_synthetic_seed":                      stage_2_synthetic_seed,         # [TC-PIPE-OUT-09]
            "kmeans_iteration_count":                     stage_2_iteration_count,        # [TC-PIPE-OUT-10]
            # Stage 3 — conjunctive median
            "delta_diff_0_to_1":                          stage_3_diff_0_to_1,            # [TC-PIPE-OUT-11]
            "delta_diff_1_to_2":                          stage_3_diff_1_to_2,            # [TC-PIPE-OUT-12]
            "delta_conjunctive_median":                   stage_3_conjunctive_median,     # [TC-PIPE-OUT-13]
            "delta_extrapolated":                         stage_3_conjunctive_median,     # [TC-PIPE-OUT-14]
            # Stage 4 — float64 Layer A  ← written to tandem_interface.synthetic_centroid_velocity_subcomponent
            "velocity_synthetic_position":                stage_4_synthetic_position,     # [TC-PIPE-OUT-15]
            "velocity_real_centroid_count":               stage_4_real_centroid_count,    # [TC-PIPE-OUT-16]
            "velocity_per_centroid_signed_tensions":      stage_4_signed_tensions,        # [TC-PIPE-OUT-17]
            "velocity_per_centroid_squared_tensions":     stage_4_squared_tensions,       # [TC-PIPE-OUT-18]
            "velocity_sum_of_squared_tensions":           stage_4_sum_squared,            # [TC-PIPE-OUT-19]
            "velocity_cumulative_magnitude":              stage_4_velocity_day_frac,      # [TC-PIPE-OUT-20] NOVEL
            "velocity_in_minutes":                        stage_4_velocity_minutes,       # [TC-PIPE-OUT-21]
            "velocity_tension_symmetry_indicator":        stage_4_symmetry_indicator,     # [TC-PIPE-OUT-22]
            "velocity_mean_absolute_tension":             stage_4_mean_abs_tension,       # [TC-PIPE-OUT-23]
            "velocity_normalized":                        stage_4_velocity_normalized,    # [TC-PIPE-OUT-24] NOVEL
            "velocity_normalized_sqrt_n_reference":       stage_4_velocity_norm_sqrt_n_ref,  # [TC-PIPE-OUT-25]
            "velocity_normalized_deviation_from_sqrt_n":  stage_4_velocity_norm_deviation,   # [TC-PIPE-OUT-26]
            # Stage 4 — Decimal Layer B (50-figure strings)
            "velocity_decimal_50fig":                     stage_4_decimal_velocity_50fig,    # [TC-PIPE-OUT-27]
            "velocity_decimal_minutes_50fig":             stage_4_decimal_vel_min_50fig,     # [TC-PIPE-OUT-28]
            "velocity_decimal_symmetry_50fig":            stage_4_decimal_symmetry_50fig,    # [TC-PIPE-OUT-29]
            "velocity_decimal_normalized_50fig":          stage_4_decimal_normalized_50fig,  # [TC-PIPE-OUT-30]
            "velocity_decimal_norm_deviation_50fig":      stage_4_decimal_norm_dev_50fig,    # [TC-PIPE-OUT-31]
        }


# ── HF ROUTING TABLE ──────────────────────────────────────────────────────────
# Phase_1 transition has THREE rules:
#   [HF-T1] checkpoint_between       → tandem_interface.results.chk_btwn_1
#   [HF-T2] thfpi_group1_pipeline    → tandem_interface.thfpi_group1
#   [HF-T3] TC-PIPE (4-stage+vel)    → tandem_interface.thfpi_group1_consolidated

_TC_PIPE = TandemCalc.thfpi_probe_pair_utc_timestamps_to_conjunctive_median_delta_via_day_normalized_synthetic_midpoint_seeded_kmeans_with_synthetic_centroid_tension_velocity

HF = {
    "Phase_1": {
        "on_entry": [{"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","dst":"tandem_interface.results.chk_entry_1","fn":TandemCalc.checkpoint_entry,"op_str":"ENTRY","note":"HF Entry scan_event_1"}],
        "on_exit":  [{"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","dst":"tandem_interface.results.chk_exit_1","fn":TandemCalc.checkpoint_exit,"op_str":"EXIT","note":"HF Exit scan_event_1"}],
        "transition": [
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","dst":"tandem_interface.results.chk_btwn_1","fn":TandemCalc.checkpoint_between,"op_str":"BETWEEN","note":"HF Transit checkpoint"},          # [HF-T1]
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","dst":"tandem_interface.thfpi_group1","fn":TandemCalc.thfpi_group1_pipeline,"op_str":"THFPI_COMPOSABLE","note":"THFPI Group 1 composable TC-N TC-K TC-C"},  # [HF-T2]
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_1.scan_timestamp_utc","dst":"tandem_interface.thfpi_group1_consolidated","fn":_TC_PIPE,"op_str":"THFPI_CONSOLIDATED_WITH_VELOCITY","note":"THFPI Group 1 TC-PIPE S1+S2+S3+S4"},  # [HF-T3]
        ],
    },
    "Phase_2": {
        "on_entry":   [{"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_2.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_2.scan_timestamp_utc","dst":"tandem_interface.results.chk_entry_2","fn":TandemCalc.checkpoint_entry,"op_str":"ENTRY","note":"HF Entry scan_event_2"}],
        "on_exit":    [{"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_2.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_2.scan_timestamp_utc","dst":"tandem_interface.results.chk_exit_2","fn":TandemCalc.checkpoint_exit,"op_str":"EXIT","note":"HF Exit scan_event_2"}],
        "transition": [
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_2.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_2.scan_timestamp_utc","dst":"tandem_interface.results.chk_btwn_2","fn":TandemCalc.checkpoint_between,"op_str":"BETWEEN","note":"HF Transit scan_event_2"},         # [HF-T3]
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_2.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_2.scan_timestamp_utc","dst":"tandem_interface.thfpi_group2","fn":TandemCalc.thfpi_group2_pipeline,"op_str":"THFPI_COMPOSABLE","note":"THFPI Group 2"},              # [HF-T4]
        ],
    },
    "Phase_3": {
        "on_entry":   [{"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_3.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_3.scan_timestamp_utc","dst":"tandem_interface.results.chk_entry_3","fn":TandemCalc.checkpoint_entry,"op_str":"ENTRY","note":"HF Entry scan_event_3"}],
        "on_exit":    [{"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_3.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_3.scan_timestamp_utc","dst":"tandem_interface.results.chk_exit_3","fn":TandemCalc.checkpoint_exit,"op_str":"EXIT","note":"HF Exit scan_event_3"}],
        "transition": [
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_3.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_3.scan_timestamp_utc","dst":"tandem_interface.results.chk_btwn_3","fn":TandemCalc.checkpoint_between,"op_str":"BETWEEN","note":"HF Transit scan_event_3"},         # [HF-T5]
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_3.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_3.scan_timestamp_utc","dst":"tandem_interface.thfpi_group3","fn":TandemCalc.thfpi_group3_pipeline,"op_str":"THFPI_COMPOSABLE","note":"THFPI Group 3"},              # [HF-T6]
        ],
    },
    "Phase_4": {
        "on_entry":   [{"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_4.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_4.scan_timestamp_utc","dst":"tandem_interface.results.chk_entry_4","fn":TandemCalc.checkpoint_entry,"op_str":"ENTRY","note":"HF Entry scan_event_4"}],
        "on_exit":    [{"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_4.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_4.scan_timestamp_utc","dst":"tandem_interface.results.chk_exit_4","fn":TandemCalc.checkpoint_exit,"op_str":"EXIT","note":"HF Exit scan_event_4"}],
        "transition": [
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_4.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_4.scan_timestamp_utc","dst":"tandem_interface.results.chk_btwn_4","fn":TandemCalc.checkpoint_between,"op_str":"BETWEEN","note":"HF Transit scan_event_4"},         # [HF-T7]
            {"direction":"merge_to_jit","src_pri":"payload.package_shipping_manifest.scan_event_4.scan_timestamp_utc","src_sec":"payload.package_shipping_manifest.scan_event_4.scan_timestamp_utc","dst":"tandem_interface.thfpi_group4","fn":TandemCalc.thfpi_group4_pipeline,"op_str":"THFPI_COMPOSABLE","note":"THFPI Group 4"},              # [HF-T8]
        ],
    },
}


# ── HFEngine ──────────────────────────────────────────────────────────────────

class HFEngine:
    last_retrieval = {}

    @staticmethod
    def _get(obj, path):
        for p in path.split("."): obj = getattr(obj, p, None)
        return obj

    @staticmethod
    def _set(obj, path, value):
        parts = path.split(".")
        for p in parts[:-1]: obj = getattr(obj, p, None)
        if obj: setattr(obj, parts[-1], value)

    @staticmethod
    def fire(node, moment, jit, pri, sec, tel):
        HFEngine.last_retrieval = {"node": node, "moment": moment, "reads": []}   # [HF-A]
        for r in HF.get(node, {}).get(moment, []):
            if r["direction"] == "merge_to_jit":
                vp  = HFEngine._get(pri, r["src_pri"])                            # [HF-B]
                vs  = HFEngine._get(sec, r["src_sec"])
                res = r["fn"](vp, vs) if r.get("fn") else (vp, vs)
                HFEngine._set(jit, r["dst"], res)
                op   = r.get("op_str", "OP")
                expr = f"{r['src_pri']}({vp}) {op} {r['src_sec']}({vs})"
                tel.record_hf(node, moment, "merge_to_jit", r["dst"], res)
                HFEngine.last_retrieval["reads"].append({                          # [HF-C]
                    "rule_note":    r.get("note", ""),
                    "src_pri_path": r["src_pri"],  "vp_retrieved": vp,
                    "src_sec_path": r["src_sec"],  "vs_retrieved": vs,
                    "fn_called":    r["fn"].__name__ if r.get("fn") else "none",
                    "result":       res, "dst_path": r["dst"], "expression": expr,
                })


# ── PipelineTelemetry ─────────────────────────────────────────────────────────

class PipelineTelemetry:
    def __init__(self):
        self.start_ns = time.perf_counter_ns(); self.uid = "TEL-CORE"
        self.entries = {}; self.durs = {}; self.skews = {}
        self.impacts = {}; self.hf_log = []; self._lock = threading.Lock()
        self.hf_event_log = []

    def enter(self, name):
        with self._lock: self.entries[name] = time.perf_counter_ns(); return f"ENT-{name}"
    def exit(self, name):
        with self._lock: self.durs[name] = time.perf_counter_ns() - self.entries.get(name, 0); return f"EXT-{name}"
    def record_hf(self, p, m, d, dst, v):
        with self._lock: self.hf_log.append((p, m, d, dst, str(v)[:120]))
    def record_hf_event(self, node, moment, reads):
        with self._lock:
            self.hf_event_log.append({
                "node": node, "moment": moment, "n_rules": len(reads),
                "fn_names":  [r.get("fn_called", "?") for r in reads],
                "dst_paths": [r.get("dst_path",  "?") for r in reads],
                "ts_ns": time.perf_counter_ns(),
            })
    def record_metrics(self, phase, skew, impact):
        with self._lock: self.skews[phase] = skew; self.impacts[phase] = impact


# ── StoryLog ──────────────────────────────────────────────────────────────────

class StoryLog:
    _buffer = []; _lock = threading.Lock()

    @classmethod
    def trace(cls, phase, narrative, ops, metrics, jit, pri, sec, tel):
        with cls._lock:
            cls._buffer.append({
                "phase": phase, "narrative": narrative,
                "ops": ops, "metrics": metrics,
                "ts_ns": time.perf_counter_ns(),
            })

    @classmethod
    def flush_buffer(cls):
        with cls._lock:
            out = list(cls._buffer); cls._buffer.clear()
        return out

    @classmethod
    def dump_objects(cls, *objs):
        for o in objs: print(o)


# ── VarTracker ────────────────────────────────────────────────────────────────

class VarTracker:
    _log = []; _step = 0; DIVIDER = "─" * 76

    @classmethod
    def snapshot(cls, label, variables: dict):
        cls._step += 1
        entry = {"step": cls._step, "label": label, "ts_ns": time.perf_counter_ns(), "vars": {}}
        lines = [f"\n{'━'*76}", f"  ◉ BP{cls._step:02d} │ {label}", cls.DIVIDER]
        for name, val in variables.items():
            display = val.to_dict() if isinstance(val, DynamicSection) else \
                      {k: v for k, v in val.__dict__.items() if not k.startswith('_')} \
                      if hasattr(val, '__dict__') and not isinstance(val, type) else val
            entry["vars"][name] = display
            lines.append(f"  {name:<68} = {repr(display)}")
        lines.append(cls.DIVIDER)
        cls._log.append(entry); print("\n".join(lines))

    @classmethod
    def delta(cls, label, variables: dict):
        cls.snapshot(label, variables)

    @classmethod
    def summary(cls):
        print(f"\n{'█'*76}\n  VARIABLE TRACKER FULL TIMELINE — {cls._step} checkpoints\n{'█'*76}")
        for e in cls._log:
            print(f"\n  ── Step {e['step']:02d}: {e['label']}")
            for name, val in e["vars"].items():
                print(f"     {name:<68} = {repr(val)}")
        print(f"\n{'█'*76}\n")


# ── ScanEventTemporalMatcher ──────────────────────────────────────────────────

class ScanEventTemporalMatcher:
    SCAN_EVENT_CROSS_PROBE_TEMPORAL_MATCH_TOLERANCE_HOURS = 2

    @staticmethod
    def extract_chronologically_ordered_scan_event_list_from_probe_manifest(manifest):
        events = []; cursor = 1
        while True:
            ev = getattr(manifest, f"scan_event_{cursor}", None)
            if ev is None: break
            events.append({
                "scan_event_sequence_number":                  ev.scan_event_sequence_number,
                "scan_event_physical_location_city_state":     ev.scan_location_city_state,
                "scan_event_timestamp_utc_string":             ev.scan_timestamp_utc,
                "scan_event_timestamp_as_comparable_datetime": datetime.datetime.strptime(ev.scan_timestamp_utc, "%Y-%m-%d %H:%M:%S"),
                "scan_event_id":                               ev.scan_event_id,
            }); cursor += 1
        return sorted(events, key=lambda e: e["scan_event_sequence_number"])

    @staticmethod
    def match_and_group_cross_probe_scan_events_by_time_period_window(primary_list, secondary_list, tolerance_hours):
        tol = datetime.timedelta(hours=tolerance_hours)
        matched = []; consumed = set()
        for pri in primary_list:
            pri_dt = pri["scan_event_timestamp_as_comparable_datetime"]
            best_sec, best_gap, best_idx = None, float("inf"), None
            for idx, sec in enumerate(secondary_list):
                if idx in consumed: continue
                gap = abs(pri_dt - sec["scan_event_timestamp_as_comparable_datetime"])
                if gap <= tol and gap.total_seconds() < best_gap:
                    best_sec = sec; best_gap = gap.total_seconds(); best_idx = idx
            if best_sec is not None:
                consumed.add(best_idx); seq = len(matched) + 1
                matched.append({
                    "matched_group_sequential_number": seq,
                    "temporal_match_tolerance_hours_used_to_form_this_group": tolerance_hours,
                    "matched_group_actual_time_gap_between_probe_events_hours": round(best_gap / 3600, 4),
                    "matched_group_primary_probe_contributing_scan_event": {
                        "scan_event_sequence_number":              pri["scan_event_sequence_number"],
                        "scan_event_physical_location_city_state": pri["scan_event_physical_location_city_state"],
                        "scan_event_timestamp_utc_string":         pri["scan_event_timestamp_utc_string"],
                        "scan_event_id":                           pri["scan_event_id"],
                    },
                    "matched_group_secondary_probe_contributing_scan_event": {
                        "scan_event_sequence_number":              best_sec["scan_event_sequence_number"],
                        "scan_event_physical_location_city_state": best_sec["scan_event_physical_location_city_state"],
                        "scan_event_timestamp_utc_string":         best_sec["scan_event_timestamp_utc_string"],
                        "scan_event_id":                           best_sec["scan_event_id"],
                    },
                })
        return matched, [], [s for i, s in enumerate(secondary_list) if i not in consumed]


# ── JITPipelineArchitecture ───────────────────────────────────────────────────

class JITPipelineArchitecture:
    class Data:
        class Models:      pass
        class Telemetry:   pass
    class Observability:   pass
    class Execution:
        class Nodes:       pass
        class Orchestrator: pass


print("Cell 1 — infrastructure loaded.")
print(f"  DynamicSection · DynamicProbe · JITObject")
print(f"  TandemCalc: TC-01..TC-05 · TC-N · TC-K · TC-C · TC-V · TC-P1..P4 · TC-PIPE")
print(f"  HF routing table · HFEngine · PipelineTelemetry · StoryLog · VarTracker")
print(f"  ScanEventTemporalMatcher · JITPipelineArchitecture")
print(f"  JITObject.tandem_interface.kmeans_centroids          ← centroid store (new)")
print(f"  JITObject.tandem_interface.synthetic_centroid_velocity_subcomponent")

Cell 1 — infrastructure loaded.
  DynamicSection · DynamicProbe · JITObject
  TandemCalc: TC-01..TC-05 · TC-N · TC-K · TC-C · TC-V · TC-P1..P4 · TC-PIPE
  HF routing table · HFEngine · PipelineTelemetry · StoryLog · VarTracker
  ScanEventTemporalMatcher · JITPipelineArchitecture
  JITObject.tandem_interface.kmeans_centroids          ← centroid store (new)
  JITObject.tandem_interface.synthetic_centroid_velocity_subcomponent


In [2]:
# @title 2: OBSERVABILITY, STORY LOGS, 3D VISUALIZATION & HF TELEMETRY GRAPH

# =============================================================================
# ACT 2: THE INSTRUMENTS — NARRATIVE STORY
# We build every lens we need to see inside the system:
#   • 3D animated telemetry scatter — skew / impact / phase trajectory
#   • Architecture network — probe→JIT→HFEngine routing flow
#   • Hierarchical infrastructure — nested payload tree
#   • HF Telemetry Dashboard — NEW: dedicated visual for every HFEngine.fire()
#     event, showing which rules fired, which TandemCalc functions were called,
#     which dst paths were written, and the THFPI pipeline result values.
# =============================================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots

class Visualizations:

    # ─────────────────────────────────────────────────────────────────────────
    # 1. ANIMATED 3D TELEMETRY SCATTER
    # ─────────────────────────────────────────────────────────────────────────
    @staticmethod
    def render_animated_telemetry(tel, pri, sec):
        phases = list(tel.skews.keys())
        if not phases: print("No telemetry phases to render."); return

        skews   = [tel.skews.get(p, 0) for p in phases]
        impacts = [tel.impacts.get(p, 0) for p in phases]
        durs    = [tel.durs.get(p, 0) / 1e6 for p in phases]  # ns → ms

        frames = []
        for i in range(1, len(phases) + 1):
            frames.append(go.Frame(
                data=[go.Scatter3d(
                    x=skews[:i], y=impacts[:i], z=durs[:i],
                    mode="lines+markers+text",
                    text=phases[:i], textposition="top center",
                    marker=dict(size=6, color=list(range(i)),
                                colorscale="Plasma", showscale=False),
                    line=dict(color="cyan", width=3),
                )],
                name=str(i),
            ))

        fig = go.Figure(
            data=[go.Scatter3d(
                x=skews[:1], y=impacts[:1], z=durs[:1],
                mode="markers+text", text=phases[:1],
                marker=dict(size=8, color="cyan"),
            )],
            frames=frames,
            layout=go.Layout(
                title="⚡ THFPI Pipeline — 3D Phase Trajectory (Skew / Impact / Duration)",
                paper_bgcolor="#0a0a1a", plot_bgcolor="#0a0a1a",
                font=dict(color="white"),
                scene=dict(
                    xaxis=dict(title="Skew (s)", backgroundcolor="#0d0d2b",
                               gridcolor="#333366", color="white"),
                    yaxis=dict(title="Impact Score", backgroundcolor="#0d0d2b",
                               gridcolor="#333366", color="white"),
                    zaxis=dict(title="Duration (ms)", backgroundcolor="#0d0d2b",
                               gridcolor="#333366", color="white"),
                    bgcolor="#0a0a1a",
                ),
                updatemenus=[dict(
                    type="buttons", showactive=False, y=1.05, x=0.5, xanchor="center",
                    buttons=[
                        dict(label="▶ Play",  method="animate",
                             args=[None, {"frame": {"duration": 600, "redraw": True}, "fromcurrent": True}]),
                        dict(label="⏸ Pause", method="animate",
                             args=[[None], {"frame": {"duration": 0}, "mode": "immediate"}]),
                    ],
                )],
            )
        )
        fig.show()

    # ─────────────────────────────────────────────────────────────────────────
    # 2. ARCHITECTURE NETWORK GRAPH
    # ─────────────────────────────────────────────────────────────────────────
    @staticmethod
    def render_architecture_network(tel, pri, sec):
        nodes = ["PRI Probe", "SEC Probe", "TandemCalc", "HFEngine", "JIT Bus",
                 "Phase_1", "Phase_2", "Phase_3", "Phase_4", "Telemetry"]
        edges = [(0,2),(1,2),(2,3),(3,4),(4,5),(4,6),(4,7),(4,8),(5,9),(6,9),(7,9),(8,9)]

        import math
        n = len(nodes)
        xs = [math.cos(2*math.pi*i/n) for i in range(n)]
        ys = [math.sin(2*math.pi*i/n) for i in range(n)]

        edge_x, edge_y = [], []
        for a, b in edges:
            edge_x += [xs[a], xs[b], None]; edge_y += [ys[a], ys[b], None]

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
                                  line=dict(color="#334499", width=1.5), hoverinfo="none"))
        colors = ["#00d2ff","#00d2ff","#ff6b6b","#ffd700","#7fff7f",
                  "#c084fc","#c084fc","#c084fc","#c084fc","#f97316"]
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="markers+text",
                                  text=nodes, textposition="top center",
                                  marker=dict(size=20, color=colors,
                                              line=dict(color="white", width=1)),
                                  hoverinfo="text"))
        fig.update_layout(
            title="🔗 THFPI Architecture — Component Routing Network",
            showlegend=False, paper_bgcolor="#0a0a1a", plot_bgcolor="#0a0a1a",
            font=dict(color="white"),
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            height=500,
        )
        fig.show()

    # ─────────────────────────────────────────────────────────────────────────
    # 3. HIERARCHICAL INFRASTRUCTURE SUNBURST
    # ─────────────────────────────────────────────────────────────────────────
    @staticmethod
    def render_hierarchical_infrastructure(jit, pri, sec):
        labels, parents, values, colors = ["THFPI"], [""], [1], ["#0a0a1a"]
        palette = {"PRI":"#00d2ff","SEC":"#ff6b6b","JIT":"#7fff7f","TI":"#ffd700","HF":"#c084fc"}

        def walk(obj, parent, color, depth=0):
            if depth > 3: return
            data = obj.to_dict() if isinstance(obj, DynamicSection) else \
                   {k: v for k, v in obj.__dict__.items() if not k.startswith("_")} if hasattr(obj, "__dict__") else {}
            for k, v in list(data.items())[:12]:
                label = f"{k[:20]}"
                if label in labels: label = f"{label}_{depth}"
                labels.append(label); parents.append(parent)
                values.append(1); colors.append(color)
                if isinstance(v, (DynamicSection,)) or (hasattr(v, '__dict__') and not isinstance(v, type)):
                    walk(v, label, color, depth+1)

        walk(pri.payload, "THFPI", palette["PRI"])
        walk(sec.payload, "THFPI", palette["SEC"])
        walk(jit.jit_metrics, "THFPI", palette["JIT"])
        walk(jit.tandem_interface, "THFPI", palette["TI"])

        fig = go.Figure(go.Sunburst(
            labels=labels, parents=parents, values=values,
            marker=dict(colors=colors, line=dict(color="black", width=0.5)),
            branchvalues="total",
        ))
        fig.update_layout(
            title="🌐 THFPI Hierarchical Infrastructure — Payload & Interface Tree",
            paper_bgcolor="#0a0a1a", font=dict(color="white"), height=600,
        )
        fig.show()

    # ─────────────────────────────────────────────────────────────────────────
    # 4. HF TELEMETRY DASHBOARD  ← NEW
    # Visualises every HFEngine.fire() event as a dedicated subplot grid:
    #   Top row    — Bar chart: rules fired per event (height = n_rules)
    #   Middle row — Timeline: fire events on a horizontal T-axis
    #   Bottom row — THFPI pipeline centroid geometry (Group 1 centroids)
    # ─────────────────────────────────────────────────────────────────────────
    @staticmethod
    def render_hf_telemetry_dashboard(tel, jit):
        fig = make_subplots(
            rows=3, cols=2,
            subplot_titles=(
                "HF Fire Events — Rules Activated per Call",
                "TandemCalc Functions Called per Event",
                "HF Event Timeline (T-offset from pipeline start)",
                "HF Destination Paths Written per Event",
                "THFPI Group 1 — Centroid Geometry",
                "THFPI Group 1 — Conjunctive Delta vs Raw Gap",
            ),
            vertical_spacing=0.14, horizontal_spacing=0.10,
        )

        events   = tel.hf_event_log
        if not events:
            print("No HF events recorded — pipeline may not have fired yet."); return

        labels   = [f"{e['node']}\n{e['moment']}" for e in events]
        n_rules  = [e["n_rules"] for e in events]
        t_offsets_ms = [(e["ts_ns"] - tel.start_ns) / 1e6 for e in events]
        fn_labels = [", ".join(set(e["fn_names"])) for e in events]
        dst_labels = ["\n".join(e["dst_paths"])[:40] for e in events]

        # ── Row 1 Left: rules per event bar ──
        bar_colors = ["#00d2ff" if n == 1 else "#ffd700" if n == 2 else "#ff6b6b" for n in n_rules]
        fig.add_trace(go.Bar(
            x=labels, y=n_rules, marker_color=bar_colors,
            text=n_rules, textposition="outside",
            name="Rules fired", showlegend=False,
        ), row=1, col=1)
        fig.update_yaxes(title_text="# Rules", row=1, col=1)

        # ── Row 1 Right: fn names per event ──
        fig.add_trace(go.Bar(
            x=labels, y=[1]*len(events), text=fn_labels,
            textposition="inside", marker_color="#c084fc",
            name="Fn called", showlegend=False,
        ), row=1, col=2)

        # ── Row 2 Left: timeline scatter ──
        fig.add_trace(go.Scatter(
            x=t_offsets_ms, y=[0]*len(events), mode="markers+text",
            text=labels, textposition="top center",
            marker=dict(size=14, color=bar_colors,
                        line=dict(color="white", width=1)),
            name="Timeline", showlegend=False,
        ), row=2, col=1)
        fig.update_xaxes(title_text="T-offset (ms)", row=2, col=1)
        fig.update_yaxes(visible=False, row=2, col=1)

        # ── Row 2 Right: dst paths per event ──
        fig.add_trace(go.Bar(
            x=labels, y=[1]*len(events), text=dst_labels,
            textposition="inside", marker_color="#7fff7f",
            name="Dst paths", showlegend=False,
        ), row=2, col=2)

        # ── Row 3: THFPI Group 1 centroid geometry ──
        _g = jit.tandem_interface.thfpi_group1 if jit and jit.tandem_interface.thfpi_group1 else {}
        if _g:
            c0   = _g.get("kmeans_centroid_0_primary",            0)
            c1   = _g.get("kmeans_centroid_1_synthetic_midpoint", 0)
            c2   = _g.get("kmeans_centroid_2_secondary",          0)
            conj = _g.get("delta_conjunctive_median",             0)
            raw  = abs(_g.get("norm_signed_delta", 0))

            # centroid number line
            fig.add_trace(go.Scatter(
                x=[c0, c1, c2],
                y=[0, 0, 0],
                mode="markers+text",
                text=[f"C0 PRIMARY<br>{c0:.6f}", f"C1 SYNTHETIC<br>{c1:.6f}", f"C2 SECONDARY<br>{c2:.6f}"],
                textposition=["bottom center","top center","bottom center"],
                marker=dict(size=[14, 18, 14],
                            color=["#00d2ff", "#ffd700", "#ff6b6b"],
                            symbol=["circle","star","circle"],
                            line=dict(color="white", width=1)),
                name="Centroids", showlegend=False,
            ), row=3, col=1)
            # gap annotation lines
            fig.add_shape(type="line", x0=c0, x1=c1, y0=0, y1=0,
                          line=dict(color="#ffd700", width=3, dash="dot"),
                          row=3, col=1, xref="x5", yref="y5")
            fig.add_shape(type="line", x0=c1, x1=c2, y0=0, y1=0,
                          line=dict(color="#ffd700", width=3, dash="dot"),
                          row=3, col=1, xref="x5", yref="y5")
            fig.update_xaxes(title_text="Normalized day-fraction", row=3, col=1)
            fig.update_yaxes(visible=False, row=3, col=1)

            # conjunctive vs raw gap bar comparison
            fig.add_trace(go.Bar(
                x=["Raw inter-probe gap", "Conjunctive median delta\n(half-gap, NOVEL)"],
                y=[raw * 1440, conj * 1440],
                marker_color=["#ff6b6b", "#ffd700"],
                text=[f"{raw*1440:.4f} min", f"{conj*1440:.4f} min"],
                textposition="outside",
                name="Delta comparison", showlegend=False,
            ), row=3, col=2)
            fig.update_yaxes(title_text="Minutes", row=3, col=2)
        else:
            fig.add_annotation(text="THFPI Group 1 not yet computed",
                               xref="paper", yref="paper", x=0.25, y=0.1,
                               showarrow=False, font=dict(color="white"))

        fig.update_layout(
            title="📡 THFPI HF Telemetry Dashboard — Every fire() Event Visualised",
            paper_bgcolor="#0a0a1a", plot_bgcolor="#0d0d2b",
            font=dict(color="white"),
            height=900,
        )
        fig.update_xaxes(color="white", gridcolor="#333366")
        fig.update_yaxes(color="white", gridcolor="#333366")
        fig.show()

JITPipelineArchitecture.Observability.Visualizations = Visualizations

In [3]:
# @title 3: FACTORY PHASE — 1 (Ingestion)
# =============================================================================
# ACT 3: THE JOURNEY BEGINS (Phase 1) — NARRATIVE STORY
# Arbitrary objects enter the dynamic factory. JIT receives its first HF values.
# We calculate initial physical skews and prove the JIT embedded functions work.
# VarTracker snapshots capture every key moment inside the factory.
# =============================================================================

class Factory_Phase_1:
    @staticmethod
    def execute(jit, pri, sec, tel):
        Log = JITPipelineArchitecture.Observability.StoryLog
        VT  = JITPipelineArchitecture.Observability.VarTracker

        # [1] tel enters Phase_1 — starts the phase clock.
        uid = tel.enter("Phase_1")
        setattr(jit, 'phase_1', DynamicSection())
        setattr(pri, 'phase_1', DynamicSection())
        setattr(sec, 'phase_1', DynamicSection())

        # ── BREAKPOINT 1 ── Phase entry, probes and JIT constructed ──────────
        VT.snapshot("PHASE 1 ENTRY — jit, pri, sec constructed; tel clock started", {
            "phase_1_entry_uid":            uid,
            "jit_uid":                      jit.uid,
            "pri_uid":                      pri.uid,
            "sec_uid":                      sec.uid,
            "pri_payload_calc_value":       pri.payload.calc_value,
            "sec_payload_calc_value":       sec.payload.calc_value,
            "pri_payload_step1_id":         pri.payload.step1_id,
            "pri_payload_step1_time":       pri.payload.step1_time,
            "sec_payload_step1_id":         sec.payload.step1_id,
            "sec_payload_step1_time":       sec.payload.step1_time,
            "jit_tandem_interface_results": jit.tandem_interface.results,
        })

        # ◀◀ THFPI ACTIVATION POINT 1/3 — on_entry fires here
        HFEngine.fire("Phase_1", "on_entry", jit, pri, sec, tel)
        # [TEL-HF2] Record this fire() event in the structured telemetry log.
        tel.record_hf_event("Phase_1", "on_entry", HFEngine.last_retrieval["reads"])

        # ── BREAKPOINT 2 ── HF on_entry tandem retrieval + full payload ──────
        _ledger = HFEngine.last_retrieval
        _r0     = _ledger["reads"][0] if _ledger["reads"] else {}
        VT.snapshot("HF on_entry — TANDEM RETRIEVAL + FULL PROBE PAYLOAD", {
            "hf_on_entry_node":                             _ledger.get("node"),
            "hf_on_entry_moment":                           _ledger.get("moment"),
            "hf_on_entry_src_pri_path":                     _r0.get("src_pri_path"),
            "hf_on_entry_vp_retrieved":                     _r0.get("vp_retrieved"),
            "hf_on_entry_src_sec_path":                     _r0.get("src_sec_path"),
            "hf_on_entry_vs_retrieved":                     _r0.get("vs_retrieved"),
            "hf_on_entry_fn_called":                        _r0.get("fn_called"),
            "hf_on_entry_result_on_tandem_interface":       _r0.get("result"),
            "hf_on_entry_dst_path_written":                 _r0.get("dst_path"),
            "hf_on_entry_expression":                       _r0.get("expression"),
            "── PRI PROBE ACCESSIBLE DATA ──":              "─" * 20,
            "pri_probe_calc_value":                         pri.payload.calc_value,
            "pri_probe_step1_id":                           pri.payload.step1_id,
            "pri_probe_step1_time":                         pri.payload.step1_time,
            "pri_probe_step2_id":                           pri.payload.step2_id,
            "pri_probe_step3_id":                           pri.payload.step3_id,
            "pri_probe_step4_id":                           pri.payload.step4_id,
            "pri_probe_manifest_tracking_id":               pri.payload.package_shipping_manifest.package_tracking_id,
            "pri_probe_manifest_scan_event_1":              pri.payload.package_shipping_manifest.scan_event_1,
            "pri_probe_manifest_scan_event_2":              pri.payload.package_shipping_manifest.scan_event_2,
            "pri_probe_manifest_scan_event_3":              pri.payload.package_shipping_manifest.scan_event_3,
            "pri_probe_hf_on_entry_expr_stamped":           pri.payload.hf_expr_Phase_1_on_entry,
            "── SEC PROBE ACCESSIBLE DATA ──":              "─" * 20,
            "sec_probe_calc_value":                         sec.payload.calc_value,
            "sec_probe_step1_id":                           sec.payload.step1_id,
            "sec_probe_step1_time":                         sec.payload.step1_time,
            "sec_probe_manifest_tracking_id":               sec.payload.package_shipping_manifest.package_tracking_id,
            "sec_probe_manifest_scan_event_1":              sec.payload.package_shipping_manifest.scan_event_1,
            "sec_probe_hf_on_entry_expr_stamped":           sec.payload.hf_expr_Phase_1_on_entry,
            "── JIT STATE ──":                              "─" * 20,
            "jit_tandem_interface_results_after_on_entry":  jit.tandem_interface.results,
            "tel_hf_log":                                   tel.hf_log,
        })

        # [4] stamp phase events
        pri.phase_1.event = "Ingestion_A_Mapped"
        sec.phase_1.event = "Ingestion_B_Mapped"
        jit.phase_1.entry_uid = uid

        # [5] lift calc values
        primary_probe_phase_calculation_value   = getattr(pri.payload, 'calc_value', 0)
        secondary_probe_phase_calculation_value = getattr(sec.payload, 'calc_value', 0)
        primary_probe_belt_step_1_scan_id       = getattr(pri.payload, 'step1_id', 'A0X')

        # ── BREAKPOINT 3 ── Probe values retrieved ────────────────────────────
        VT.snapshot("PROBE VALUES RETRIEVED FROM BOTH PROBE PAYLOADS", {
            "primary_probe_phase_calculation_value":        primary_probe_phase_calculation_value,
            "secondary_probe_phase_calculation_value":      secondary_probe_phase_calculation_value,
            "primary_probe_belt_step_1_scan_id":            primary_probe_belt_step_1_scan_id,
            "pri_probe_phase_1_event_label":                pri.phase_1.event,
            "sec_probe_phase_1_event_label":                sec.phase_1.event,
            "jit_phase_1_entry_uid":                        jit.phase_1.entry_uid,
        })

        # [7] JIT embedded regex
        phase_1_embedded_regex_numeric_extraction_result = jit.evaluate_regex(r"\d+", primary_probe_belt_step_1_scan_id)

        # ── BREAKPOINT 4 ── After JIT regex ───────────────────────────────────
        VT.delta("AFTER JIT evaluate_regex — numeric extraction from belt step 1 scan id", {
            "phase_1_embedded_regex_numeric_extraction_result": phase_1_embedded_regex_numeric_extraction_result,
            "primary_probe_belt_step_1_scan_id_input":          primary_probe_belt_step_1_scan_id,
            "jit_metrics_embedded_results":                     jit.jit_metrics.embedded_results,
        })

        # [8] manual TandemCalc product
        phase_1_manual_tandem_cross_probe_product_result = TandemCalc.manual_phase_calc(
            primary_probe_phase_calculation_value, secondary_probe_phase_calculation_value)

        # ── BREAKPOINT 5 ── After manual tandem calc ──────────────────────────
        VT.delta("AFTER TandemCalc.manual_phase_calc — cross-probe product computed", {
            "phase_1_manual_tandem_cross_probe_product_result": phase_1_manual_tandem_cross_probe_product_result,
            "primary_probe_phase_calculation_value":            primary_probe_phase_calculation_value,
            "secondary_probe_phase_calculation_value":          secondary_probe_phase_calculation_value,
            "formula_string": f"{primary_probe_phase_calculation_value} * {secondary_probe_phase_calculation_value} = {phase_1_manual_tandem_cross_probe_product_result}",
        })

        # [9] store into jit
        jit.jit_metrics.embedded_results.phase_1_embedded_regex_numeric_extraction_result = phase_1_embedded_regex_numeric_extraction_result
        jit.tandem_interface.results.phase_1_manual_tandem_cross_probe_product_result      = phase_1_manual_tandem_cross_probe_product_result

        # ── BREAKPOINT 6 ── After JIT stores results ──────────────────────────
        VT.delta("AFTER JIT stores regex and tandem product results into jit storage", {
            "jit_metrics_embedded_results":             jit.jit_metrics.embedded_results,
            "jit_tandem_interface_results":             jit.tandem_interface.results,
        })

        # [10-15] skew + impact
        primary_probe_belt_step1_wall_clock_offset_seconds   = getattr(pri.payload, 'step1_time', 0)
        secondary_probe_belt_step1_wall_clock_offset_seconds = getattr(sec.payload, 'step1_time', 0)
        cross_probe_belt_step1_absolute_wall_clock_duration_seconds = abs(
            primary_probe_belt_step1_wall_clock_offset_seconds - secondary_probe_belt_step1_wall_clock_offset_seconds)
        combined_probe_phase_calculation_value_mass = (
            getattr(pri.payload, 'calc_value', 1) + getattr(sec.payload, 'calc_value', 1))
        tandem_product_carried_forward_as_impact_scale_factor = phase_1_manual_tandem_cross_probe_product_result
        signed_cross_probe_belt_step1_arrival_time_skew_seconds = TandemCalc.subtract(
            primary_probe_belt_step1_wall_clock_offset_seconds,
            secondary_probe_belt_step1_wall_clock_offset_seconds)
        phase_1_impact_score = (
            (cross_probe_belt_step1_absolute_wall_clock_duration_seconds * combined_probe_phase_calculation_value_mass)
            + (abs(signed_cross_probe_belt_step1_arrival_time_skew_seconds) * tandem_product_carried_forward_as_impact_scale_factor))

        # ── BREAKPOINT 7 ── All metrics computed ──────────────────────────────
        VT.snapshot("ALL PHASE 1 METRICS COMPUTED — ready to commit to telemetry", {
            "primary_probe_belt_step1_wall_clock_offset_seconds":            primary_probe_belt_step1_wall_clock_offset_seconds,
            "secondary_probe_belt_step1_wall_clock_offset_seconds":          secondary_probe_belt_step1_wall_clock_offset_seconds,
            "cross_probe_belt_step1_absolute_wall_clock_duration_seconds":   cross_probe_belt_step1_absolute_wall_clock_duration_seconds,
            "combined_probe_phase_calculation_value_mass":                   combined_probe_phase_calculation_value_mass,
            "tandem_product_carried_forward_as_impact_scale_factor":         tandem_product_carried_forward_as_impact_scale_factor,
            "signed_cross_probe_belt_step1_arrival_time_skew_seconds":       signed_cross_probe_belt_step1_arrival_time_skew_seconds,
            "phase_1_impact_score":                                          phase_1_impact_score,
            "phase_1_impact_score_formula_string": (
                f"({cross_probe_belt_step1_absolute_wall_clock_duration_seconds}"
                f"*{combined_probe_phase_calculation_value_mass})"
                f" + ({abs(signed_cross_probe_belt_step1_arrival_time_skew_seconds)}"
                f"*{tandem_product_carried_forward_as_impact_scale_factor})"
                f" = {phase_1_impact_score}"
            ),
        })

        # [16] commit to telemetry
        tel.record_metrics("Phase_1", signed_cross_probe_belt_step1_arrival_time_skew_seconds, phase_1_impact_score)

        # [17] story log
        Log.trace("Phase_1",
            "Acquisition of values from PRI and SEC. Execution of embedded JIT regex. Manual execution of separate Tandem function complete.",
            [f"JIT Embedded Regex r'\\d+' on '{primary_probe_belt_step_1_scan_id}' -> {phase_1_embedded_regex_numeric_extraction_result}",
             f"Tandem Manual phase_calc({primary_probe_phase_calculation_value}, {secondary_probe_phase_calculation_value}) -> {phase_1_manual_tandem_cross_probe_product_result}"],
            {"Skew": signed_cross_probe_belt_step1_arrival_time_skew_seconds,
             "Manual_Tandem": phase_1_manual_tandem_cross_probe_product_result,
             "Impact": phase_1_impact_score},
            jit, pri, sec, tel)

        # ◀◀ THFPI ACTIVATION POINT 2/3 — on_exit fires here
        HFEngine.fire("Phase_1", "on_exit", jit, pri, sec, tel)
        tel.record_hf_event("Phase_1", "on_exit", HFEngine.last_retrieval["reads"])

        # ── BREAKPOINT 8 ── HF on_exit tandem retrieval + full payload ────────
        _ledger_exit = HFEngine.last_retrieval
        _r0e         = _ledger_exit["reads"][0] if _ledger_exit["reads"] else {}
        VT.snapshot("HF on_exit — TANDEM RETRIEVAL + FULL PROBE + JIT STATE", {
            "hf_on_exit_node":                              _ledger_exit.get("node"),
            "hf_on_exit_moment":                            _ledger_exit.get("moment"),
            "hf_on_exit_src_pri_path":                      _r0e.get("src_pri_path"),
            "hf_on_exit_vp_retrieved":                      _r0e.get("vp_retrieved"),
            "hf_on_exit_src_sec_path":                      _r0e.get("src_sec_path"),
            "hf_on_exit_vs_retrieved":                      _r0e.get("vs_retrieved"),
            "hf_on_exit_fn_called":                         _r0e.get("fn_called"),
            "hf_on_exit_result_on_tandem_interface":        _r0e.get("result"),
            "hf_on_exit_dst_path_written":                  _r0e.get("dst_path"),
            "hf_on_exit_expression":                        _r0e.get("expression"),
            "── PRI PROBE ──":                              "─" * 20,
            "pri_probe_hf_on_exit_expr_stamped":            pri.payload.hf_expr_Phase_1_on_exit,
            "── SEC PROBE ──":                              "─" * 20,
            "sec_probe_hf_on_exit_expr_stamped":            sec.payload.hf_expr_Phase_1_on_exit,
            "── JIT STATE ──":                              "─" * 20,
            "jit_tandem_interface_results_after_on_exit":   jit.tandem_interface.results,
            "tel_skews_dict":                               tel.skews,
            "tel_impacts_dict":                             tel.impacts,
            "tel_hf_log":                                   tel.hf_log,
        })

        # [19] stop phase clock
        tel.exit("Phase_1")
        return jit, pri, sec

JITPipelineArchitecture.Execution.Nodes.Phase_1 = Factory_Phase_1

In [4]:
# @title 4: FACTORY PHASE — 2 (Transformation)
# =============================================================================
# ACT 4: DIVERGENCE (Phase 2) - NARRATIVE STORY
# Packages transition through transformation layers. We use tandem subtractions
# to measure their distance apart and store it universally dynamically.
# =============================================================================

class Factory_Phase_2:
    @staticmethod
    def execute(jit, pri, sec, tel):
        Log, Arch = JITPipelineArchitecture.Observability.StoryLog, JITPipelineArchitecture
        uid = tel.enter("Phase_2")

        setattr(jit, 'phase_2', DynamicSection())
        setattr(pri, 'phase_2', DynamicSection())
        setattr(sec, 'phase_2', DynamicSection())

        # HF Matrix automatically fires the checkpoint_entry Tandem function!
        HFEngine.fire("Phase_2", "on_entry", jit, pri, sec, tel)

        pri.phase_2.event = "Transform_A_Logged"
        sec.phase_2.event = "Transform_B_Logged"
        jit.phase_2.entry_uid = uid

        # 115: Retrieve the next set of values from the probes.
        val_pri = getattr(pri.payload, 'calc_value', 0)
        val_sec = getattr(sec.payload, 'calc_value', 0)
        shipping_id_sec = getattr(sec.payload, 'step2_id', 'B0Y')

        # 116: Select the JIT object and invoke the embedded linear regex function.
        regex_result = jit.evaluate_regex(r"\d+", shipping_id_sec)

        # 117: Call the manual Tandem function explicitly inside the factory stage.
        manual_tandem_result = TandemCalc.manual_phase_calc(val_pri, val_sec)

        # 118: Store the manual results into the JIT architecture.
        jit.jit_metrics.embedded_results.phase_2_regex = regex_result
        jit.tandem_interface.results.phase_2_manual_calc = manual_tandem_result

        off_p = getattr(pri.payload, 'step2_time', 0)
        off_s = getattr(sec.payload, 'step2_time', 0)
        dur_p = off_p - getattr(pri.payload, 'step1_time', 0)
        dur_s = off_s - getattr(sec.payload, 'step1_time', 0)
        wall_duration = abs(dur_p + dur_s)
        obj_mass = getattr(pri.payload, 'calc_value', 1)
        tandem_conn = abs(manual_tandem_result) if manual_tandem_result != 0 else 1
        skew = TandemCalc.subtract(off_p, off_s)
        impact = (wall_duration * obj_mass) / tandem_conn
        tel.record_metrics("Phase_2", skew, impact)

        narr = f"Objects route through active divergence sectors (Running in PARALLEL with Phase 3). Execution of JIT Embedded Regex and Tandem Interface manual phase function complete."
        ops =[f"JIT Embedded Regex r'\d+' on '{shipping_id_sec}' -> {regex_result}", f"Tandem Manual phase_calc({val_pri}, {val_sec}) -> {manual_tandem_result}"]
        Log.trace("Phase_2", narr, ops, {"Route_Drift": skew, "Impact": impact}, jit, pri, sec, tel)

        # HF Matrix automatically fires the checkpoint_exit Tandem function!
        HFEngine.fire("Phase_2", "on_exit", jit, pri, sec, tel)

        tel.exit("Phase_2")
        return jit, pri, sec

JITPipelineArchitecture.Execution.Nodes.Phase_2 = Factory_Phase_2

<>:52: SyntaxWarning: invalid escape sequence '\d'
<>:52: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_277/3320084938.py:52: SyntaxWarning: invalid escape sequence '\d'
  ops =[f"JIT Embedded Regex r'\d+' on '{shipping_id_sec}' -> {regex_result}", f"Tandem Manual phase_calc({val_pri}, {val_sec}) -> {manual_tandem_result}"]


In [5]:
# @title 5: FACTORY PHASE — 3 (Synthesis)
# =============================================================================
# ACT 5: SYNTHESIS (Phase 3) - NARRATIVE STORY
# The objects logically meet back up. We dynamically calculate total accumulated
# mass (addition) and prove cross-verification using dynamic property lookups.
# =============================================================================

class Factory_Phase_3:
    @staticmethod
    def execute(jit, pri, sec, tel):
        Log, Arch = JITPipelineArchitecture.Observability.StoryLog, JITPipelineArchitecture
        uid = tel.enter("Phase_3")

        setattr(jit, 'phase_3', DynamicSection())
        setattr(pri, 'phase_3', DynamicSection())
        setattr(sec, 'phase_3', DynamicSection())

        # HF Matrix automatically fires the checkpoint_entry Tandem function!
        HFEngine.fire("Phase_3", "on_entry", jit, pri, sec, tel)

        pri.phase_3.event = "Sync_A_Aligned"
        sec.phase_3.event = "Sync_B_Aligned"
        jit.phase_3.entry_uid = uid

        # 119: Retrieve values from the probe payloads.
        val_pri = getattr(pri.payload, 'calc_value', 0)
        val_sec = getattr(sec.payload, 'calc_value', 0)
        shipping_time_pri = getattr(pri.payload, 'step3_time', 0.0)

        # 120: Select the JIT object and invoke the embedded linear regex function.
        regex_result = jit.evaluate_regex(r"\d+", shipping_time_pri)

        # 121: Call the manual Tandem function explicitly.
        manual_tandem_result = TandemCalc.manual_phase_calc(val_pri, val_sec)

        # 122: Store results into the JIT architecture.
        jit.jit_metrics.embedded_results.phase_3_regex = regex_result
        jit.tandem_interface.results.phase_3_manual_calc = manual_tandem_result

        off_p = getattr(pri.payload, 'step3_time', 0)
        off_s = getattr(sec.payload, 'step3_time', 0)
        dur_p = off_p - getattr(pri.payload, 'step2_time', 0)
        dur_s = off_s - getattr(sec.payload, 'step2_time', 0)
        wall_duration = abs(dur_p + dur_s)
        obj_mass = getattr(pri.payload, 'calc_value', 1)
        tandem_conn = manual_tandem_result
        skew = TandemCalc.subtract(off_p, off_s)
        impact = (wall_duration * obj_mass) + (tandem_conn * abs(skew))
        tel.record_metrics("Phase_3", skew, impact)

        narr = "Probes synchronize states (Running in PARALLEL with Phase 2). Linear Regex captures shipping timestamps. Manual execution of tandem formula explicitly recorded."
        ops =[f"JIT Embedded Regex r'\d+' on '{shipping_time_pri}' -> {regex_result}", f"Tandem Manual phase_calc({val_pri}, {val_sec}) -> {manual_tandem_result}"]
        Log.trace("Phase_3", narr, ops, {"Sync_Window": skew, "Manual_Tandem_Result": manual_tandem_result, "Impact": impact}, jit, pri, sec, tel)

        # HF Matrix automatically fires the checkpoint_exit Tandem function!
        HFEngine.fire("Phase_3", "on_exit", jit, pri, sec, tel)

        tel.exit("Phase_3")
        return jit, pri, sec

JITPipelineArchitecture.Execution.Nodes.Phase_3 = Factory_Phase_3

<>:52: SyntaxWarning: invalid escape sequence '\d'
<>:52: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_277/1896917041.py:52: SyntaxWarning: invalid escape sequence '\d'
  ops =[f"JIT Embedded Regex r'\d+' on '{shipping_time_pri}' -> {regex_result}", f"Tandem Manual phase_calc({val_pri}, {val_sec}) -> {manual_tandem_result}"]


In [6]:
# @title 6: FACTORY PHASE — 4 (Terminus)
# =============================================================================
# ACT 6: TERMINUS (Phase 4) - NARRATIVE STORY
# The factory journey ends. Final tandem multiplications are run dynamically,
# the objects are sealed, and the JIT memory object is preserved for parsing.
# =============================================================================

class Factory_Phase_4:
    @staticmethod
    def execute(jit, pri, sec, tel):
        Log, Arch = JITPipelineArchitecture.Observability.StoryLog, JITPipelineArchitecture
        uid = tel.enter("Phase_4")

        setattr(jit, 'phase_4', DynamicSection())
        setattr(pri, 'phase_4', DynamicSection())
        setattr(sec, 'phase_4', DynamicSection())

        # HF Matrix automatically fires the checkpoint_entry Tandem function!
        HFEngine.fire("Phase_4", "on_entry", jit, pri, sec, tel)

        pri.phase_4.event = "Outbound_A_Secured"
        sec.phase_4.event = "Outbound_B_Secured"
        jit.phase_4.entry_uid = uid

        # 123: Retrieve final values from the probes.
        val_pri = getattr(pri.payload, 'calc_value', 0)
        val_sec = getattr(sec.payload, 'calc_value', 0)
        shipping_id_sec = getattr(sec.payload, 'step4_id', 'B0Y')

        # 124: Select the JIT object and invoke the embedded linear regex function.
        regex_result = jit.evaluate_regex(r"\d+", shipping_id_sec)

        # 125: Call the manual Tandem function explicitly to seal the phase.
        manual_tandem_result = TandemCalc.manual_phase_calc(val_pri, val_sec)

        # 126: Store the final manual results into the JIT architecture.
        jit.jit_metrics.embedded_results.phase_4_regex = regex_result
        jit.tandem_interface.results.phase_4_manual_calc = manual_tandem_result

        off_p = getattr(pri.payload, 'step4_time', 0)
        off_s = getattr(sec.payload, 'step4_time', 0)
        dur_p = off_p - getattr(pri.payload, 'step3_time', 0)
        dur_s = off_s - getattr(sec.payload, 'step3_time', 0)
        wall_duration = abs(dur_p + dur_s)
        obj_mass = getattr(pri.payload, 'calc_value', 1)
        tandem_conn = manual_tandem_result
        skew = TandemCalc.subtract(off_p, off_s)
        impact = (wall_duration * obj_mass) + tandem_conn - abs(skew)
        tel.record_metrics("Phase_4", skew, impact)

        narr = "Final tandem lock achieved. Extraction of final shipping IDs complete. Execution of separate manual tandem function securely tracked."
        ops =[f"JIT Embedded Regex r'\d+' on '{shipping_id_sec}' -> {regex_result}", f"Tandem Manual phase_calc({val_pri}, {val_sec}) -> {manual_tandem_result}"]
        Log.trace("Phase_4", narr, ops, {"Final_Lock": manual_tandem_result, "Impact": impact}, jit, pri, sec, tel)

        # HF Matrix automatically fires the checkpoint_exit Tandem function!
        HFEngine.fire("Phase_4", "on_exit", jit, pri, sec, tel)

        Log.trace("Phase_4", f"Terminus Lock[{manual_tandem_result}]. Probes Released safely past bounds.",[], {}, jit, pri, sec, tel)

        tel.exit("Phase_4")
        return jit, pri, sec

JITPipelineArchitecture.Execution.Nodes.Phase_4 = Factory_Phase_4

<>:52: SyntaxWarning: invalid escape sequence '\d'
<>:52: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_277/965404433.py:52: SyntaxWarning: invalid escape sequence '\d'
  ops =[f"JIT Embedded Regex r'\d+' on '{shipping_id_sec}' -> {regex_result}", f"Tandem Manual phase_calc({val_pri}, {val_sec}) -> {manual_tandem_result}"]


In [7]:
# @title 7: FACTORY ORCHESTRATOR
# =============================================================================
# ACT 7: THE CONDUCTOR — NARRATIVE STORY
# The Orchestrator wires all phases together and drives the full pipeline.
# After Phase 1 entry/exit, the transition fire() executes TWO rules:
#   Rule 1 — TandemCalc.checkpoint_between  → chk_btwn_1 token
#   Rule 2 — TandemCalc.thfpi_group1_pipeline → normalize + k-means (k=3,
#             synthetic midpoint) + conjunctive median delta; full result dict
#             written to tandem_interface.thfpi_group1 in ONE fire() call.
# =============================================================================

class FactoryOrchestrator:
    @staticmethod
    def run(pri, sec):
        Arch  = JITPipelineArchitecture
        Log   = Arch.Observability.StoryLog
        Nodes = Arch.Execution.Nodes

        tel = Arch.Data.Telemetry.PipelineTelemetry()
        tel.start_ns  = time.perf_counter_ns()
        tel.uid       = f"PLINE-{datetime.datetime.now().strftime('%H%M%S%f')}"
        pri.telemetry = tel

        Log.trace("Orchestrator", "System Idle. Zero objects in corridor. Initialization imminent.", [], {}, None, None, None, tel)

        jit = Arch.Data.Models.JITObject()
        Log.trace("Orchestrator", f"JIT Engine spawned: {jit.uid}.", [], {}, jit, None, None, tel)
        Log.trace("Orchestrator", f"Probes injected: {pri.uid} and {sec.uid} carrying initial static values.", [], {}, jit, pri, sec, tel)

        # ─────────────────────────────────────────────────────────────────────
        # [ORC-M] MANIFEST CONVERSION
        # Converts raw_manifest_data dicts into individually named DynamicSection
        # scan_event_N attributes addressable by the HF pipeline rule dot-path.
        # ─────────────────────────────────────────────────────────────────────
        for probe, raw in [(pri, pri.payload.raw_manifest_data),
                           (sec, sec.payload.raw_manifest_data)]:
            probe.payload.package_shipping_manifest = DynamicSection()
            probe.payload.package_shipping_manifest.package_tracking_id = raw["package_tracking_id"]
            for ev_dict in raw["scan_event_history"]:
                n  = ev_dict["scan_event_sequence_number"]
                ev = DynamicSection()
                ev.scan_event_sequence_number = n
                ev.scan_location_city_state   = ev_dict["scan_location_city_state"]
                ev.scan_timestamp_utc         = ev_dict["scan_timestamp_utc"]
                ev.scan_event_id              = ev_dict["scan_event_id"]
                setattr(probe.payload.package_shipping_manifest, f"scan_event_{n}", ev)  # [ORC-M3]

        # ─────────────────────────────────────────────────────────────────────
        # [ORC-T] TEMPORAL MATCHING
        # ─────────────────────────────────────────────────────────────────────
        pri_events = ScanEventTemporalMatcher \
            .extract_chronologically_ordered_scan_event_list_from_probe_manifest(
                pri.payload.package_shipping_manifest)
        sec_events = ScanEventTemporalMatcher \
            .extract_chronologically_ordered_scan_event_list_from_probe_manifest(
                sec.payload.package_shipping_manifest)

        matched_groups, pri_unmatched, sec_unmatched = \
            ScanEventTemporalMatcher \
            .match_and_group_cross_probe_scan_events_by_time_period_window(
                pri_events, sec_events,
                ScanEventTemporalMatcher.SCAN_EVENT_CROSS_PROBE_TEMPORAL_MATCH_TOLERANCE_HOURS)

        # [ORC-T3] Write all grouping results to tandem_interface.
        jit.tandem_interface.cross_probe_scan_event_time_period_matched_groups                  = matched_groups
        jit.tandem_interface.primary_probe_scan_events_with_no_secondary_match_within_tolerance = pri_unmatched
        jit.tandem_interface.secondary_probe_scan_events_with_no_primary_match_within_tolerance = sec_unmatched

        Log.trace("Orchestrator",
            f"THFPI temporal matching complete. {len(matched_groups)} matched groups written to tandem_interface.",
            [f"Group {g['matched_group_sequential_number']}: "
             f"{g['matched_group_primary_probe_contributing_scan_event']['scan_event_physical_location_city_state']} ↔ "
             f"{g['matched_group_secondary_probe_contributing_scan_event']['scan_event_physical_location_city_state']}  "
             f"({g['matched_group_actual_time_gap_between_probe_events_hours']} hr)"
             for g in matched_groups],
            {}, jit, pri, sec, tel)

        # ─────────────────────────────────────────────────────────────────────
        # PHASE 1 — on_entry and on_exit fire inside Factory_Phase_1.execute()
        # ─────────────────────────────────────────────────────────────────────
        Nodes.Phase_1.execute(jit, pri, sec, tel)

        # ◀◀ THFPI ACTIVATION POINT 3/3 — transition fires here
        # Rule 1: checkpoint_between      → tandem_interface.results.chk_btwn_1
        # Rule 2: thfpi_group1_pipeline(vp, vs) → tandem_interface.thfpi_group1
        HFEngine.fire("Phase_1", "transition", jit, pri, sec, tel)
        tel.record_hf_event("Phase_1", "transition", HFEngine.last_retrieval["reads"])

        # ─────────────────────────────────────────────────────────────────────
        # [ORC-R] SURFACE RESULTS — read from tandem_interface after transition
        # ─────────────────────────────────────────────────────────────────────
        _g  = jit.tandem_interface.thfpi_group1 or {}
        _ti = jit.tandem_interface

        norm_pri  = _g.get("norm_primary",                        "n/a")
        norm_sec  = _g.get("norm_secondary",                      "n/a")
        c0        = _g.get("kmeans_centroid_0_primary",           "n/a")
        c1        = _g.get("kmeans_centroid_1_synthetic_midpoint","n/a")
        c2        = _g.get("kmeans_centroid_2_secondary",         "n/a")
        conj      = _g.get("delta_conjunctive_median",            "n/a")
        iters     = _g.get("kmeans_iteration_count",              "n/a")
        conj_min  = round(conj * 1440, 4) if isinstance(conj, float) else "n/a"

        Log.trace("Orchestrator",
            "THFPI Group 1 pipeline executed via HFEngine transition fire. "
            "TandemCalc.thfpi_group1_pipeline ran as a first-class HF embedded function. "
            "normalize_to_day_fraction → kmeans_three_centroid_synthetic(k=3) → "
            "conjunctive_median_delta all executed inside a single HFEngine.fire() call. "
            "Full result dict written to tandem_interface.thfpi_group1.",
            [
                f"norm_primary                    = {norm_pri:.10f}  (day fraction)",
                f"norm_secondary                  = {norm_sec:.10f}  (day fraction)",
                f"centroid_0  [PRIMARY]            = {c0:.10f}",
                f"centroid_1  [SYNTHETIC midpoint] = {c1:.10f}  (NOVEL)",
                f"centroid_2  [SECONDARY]          = {c2:.10f}",
                f"conjunctive median delta         = {conj:.10f}  ({conj_min} min  NOVEL)",
                f"k-means iterations               = {iters}  (0 = immediate convergence)",
            ],
            {"norm_primary": norm_pri, "norm_secondary": norm_sec, "conjunctive_median_min": conj_min},
            jit, pri, sec, tel)

        # ── THFPI LIVE INTERFACE SNAPSHOT ─────────────────────────────────────
        print("─── THFPI · TANDEM HIGH-FREQUENCY PROBE INTERFACE · LIVE SNAPSHOT ───────────")
        print("  [ LAYER 1 — HF ROUTING RESULTS ]")
        print(f"  chk_entry_1             {_ti.results.chk_entry_1}")                                                                                    # THFPI-C-01
        print(f"  chk_exit_1              {_ti.results.chk_exit_1}")                                                                                     # THFPI-C-02
        print(f"  chk_btwn_1              {_ti.results.chk_btwn_1}")                                                                                     # THFPI-C-03
        print("  [ LAYER 2 — TandemCalc.normalize_to_day_fraction (embedded HF function) ]")
        print(f"  norm primary            {_g.get('norm_primary',   0):.10f}  (day fraction)")                                                           # THFPI-C-04
        print(f"  norm secondary          {_g.get('norm_secondary', 0):.10f}  (day fraction)")                                                           # THFPI-C-05
        print(f"  norm signed delta       {_g.get('norm_signed_delta',0):.10f}  ({_g.get('norm_signed_delta',0)*1440:.4f} min)")                        # THFPI-C-06
        print(f"  day anchor              {_g.get('norm_anchor_utc_string','n/a')}  (midnight UTC — 86400-s zero-point)")                                # THFPI-C-07
        print("  [ LAYER 3 — TandemCalc.kmeans_three_centroid_synthetic (embedded HF function) ]")
        print(f"  k-means iterations      {_g.get('kmeans_iteration_count','n/a')}  (0 = immediate convergence)")                                        # THFPI-C-08
        print(f"  centroid_0  [PRIMARY]   {_g.get('kmeans_centroid_0_primary',0):.10f}  — real primary probe")                                           # THFPI-C-09
        print(f"  centroid_1  [SYNTHETIC] {_g.get('kmeans_centroid_1_synthetic_midpoint',0):.10f}  — synthetic midpoint  (NOVEL)")                       # THFPI-C-10
        print(f"  centroid_2  [SECONDARY] {_g.get('kmeans_centroid_2_secondary',0):.10f}  — real secondary probe")                                       # THFPI-C-11
        print("  [ LAYER 4 — TandemCalc.conjunctive_median_delta (embedded HF function) ]")
        print(f"  diff 0→1                {_g.get('delta_diff_0_to_1',0):.10f}  ({_g.get('delta_diff_0_to_1',0)*1440:.4f} min  primary→midpoint)")      # THFPI-C-12
        print(f"  diff 1→2                {_g.get('delta_diff_1_to_2',0):.10f}  ({_g.get('delta_diff_1_to_2',0)*1440:.4f} min  midpoint→secondary)")    # THFPI-C-13
        print(f"  conjunctive median      {_g.get('delta_conjunctive_median',0):.10f}  ({_g.get('delta_conjunctive_median',0)*1440:.4f} min  NOVEL)")    # THFPI-C-14
        print(f"  extrapolated delta      {_g.get('delta_extrapolated',0):.10f}  ({_g.get('delta_extrapolated',0)*1440:.4f} min)")                       # THFPI-C-15
        print("  [ THFPI EMBEDDED FUNCTION — TandemCalc.subtract (live from shared bus) ]")
        _sub = TandemCalc.subtract(_g.get("kmeans_centroid_2_secondary",0), _g.get("kmeans_centroid_0_primary",0))                                       # THFPI-C-16/17
        print(f"  subtract(c2, c0)        {_sub:.10f}  ({_sub*1440:.6f} min)  ← full inter-probe gap from live bus")                                    # THFPI-C-19
        print("─────────────────────────────────────────────────────────────────────────────")

        # ─────────────────────────────────────────────────────────────────────
        # PHASES 2 + 3 — parallel threads; PHASE 4 — sequential terminus
        # ─────────────────────────────────────────────────────────────────────
        Log.trace("Orchestrator", "Diverging: Phase 2 and Phase 3 in parallel threads.", [], {}, jit, pri, sec, tel)
        with concurrent.futures.ThreadPoolExecutor() as executor:
            f2 = executor.submit(Nodes.Phase_2.execute, jit, pri, sec, tel)
            f3 = executor.submit(Nodes.Phase_3.execute, jit, pri, sec, tel)
            concurrent.futures.wait([f2, f3])

        HFEngine.fire("Phase_2", "transition", jit, pri, sec, tel)
        tel.record_hf_event("Phase_2", "transition", HFEngine.last_retrieval["reads"])
        HFEngine.fire("Phase_3", "transition", jit, pri, sec, tel)
        tel.record_hf_event("Phase_3", "transition", HFEngine.last_retrieval["reads"])

        Nodes.Phase_4.execute(jit, pri, sec, tel)

        Log.trace("Orchestrator",
            "Tandem progression terminated. Yielding state-locked probes and JIT.",
            [], {}, None, pri, sec, tel)
        return jit, pri, sec

JITPipelineArchitecture.Execution.Orchestrator = FactoryOrchestrator

In [8]:
# @title 8: INVOCATION & CONFIGURATION
# =============================================================================
# ACT 8: IGNITION — NARRATIVE STORY
# We inject arbitrary data into the dynamic probes and push start.
# Output order: Colab UI Graphs FIRST, then narrative logs.
# The THFPI pipeline fires automatically via HFEngine.fire("Phase_1","transition").
# =============================================================================

from IPython.display import display, HTML

Arch = JITPipelineArchitecture

# [R1] Toggle — injects full package manifests into both probes.
INJECT_COMPLEX_MANIFEST_DATA_INTO_PROBE_PAYLOADS = True

# ─────────────────────────────────────────────────────────────────────────────
# [R2] BELT CONVEYOR STEP DATA — feeds step1_id, step1_time into each probe.
# ─────────────────────────────────────────────────────────────────────────────
PRIMARY_PROBE_BELT_CONVEYOR_STEP_DATA = [
    {"step_sequence_position": 1, "belt_station_location_name": "Inbound", "belt_step_scan_id": "A1X", "belt_step_wall_clock_offset_seconds": 0.0},
    {"step_sequence_position": 2, "belt_station_location_name": "Belt_A",  "belt_step_scan_id": "A2X", "belt_step_wall_clock_offset_seconds": 19.0},
    {"step_sequence_position": 3, "belt_station_location_name": "Conv",    "belt_step_scan_id": "A3X", "belt_step_wall_clock_offset_seconds": 36.0},
    {"step_sequence_position": 4, "belt_station_location_name": "Out",     "belt_step_scan_id": "A4X", "belt_step_wall_clock_offset_seconds": 55.0},
]
SECONDARY_PROBE_BELT_CONVEYOR_STEP_DATA = [
    {"step_sequence_position": 1, "belt_station_location_name": "Inbound", "belt_step_scan_id": "B1Y", "belt_step_wall_clock_offset_seconds": 3.0},
    {"step_sequence_position": 2, "belt_station_location_name": "Belt_B",  "belt_step_scan_id": "B2Y", "belt_step_wall_clock_offset_seconds": 32.0},
    {"step_sequence_position": 3, "belt_station_location_name": "Conv",    "belt_step_scan_id": "B3Y", "belt_step_wall_clock_offset_seconds": 41.0},
    {"step_sequence_position": 4, "belt_station_location_name": "Out",     "belt_step_scan_id": "B4Y", "belt_step_wall_clock_offset_seconds": 52.0},
]

# ─────────────────────────────────────────────────────────────────────────────
# [R3] PRIMARY PROBE MANIFEST — PKG-A102, 8 scan events, 4 location clusters.
#      Clusters: Los Angeles (1-2), Denver (3-4), Kansas City (5-6), Columbus (7-8).
#      THFPI Group 1 uses scan_event_1: 2026-03-01 08:00:00 (Los Angeles, CA).
# ─────────────────────────────────────────────────────────────────────────────
PRIMARY_PROBE_PACKAGE_MANIFEST_RAW_DATA = {
    "package_tracking_id": "PKG-A102",
    "scan_event_history": [
        {"scan_event_sequence_number": 1, "scan_location_city_state": "Los Angeles, CA", "scan_timestamp_utc": "2026-03-01 08:00:00", "scan_event_id": "SCAN-A101"},
        {"scan_event_sequence_number": 2, "scan_location_city_state": "Los Angeles, CA", "scan_timestamp_utc": "2026-03-01 08:25:00", "scan_event_id": "SCAN-A102"},
        {"scan_event_sequence_number": 3, "scan_location_city_state": "Denver, CO",      "scan_timestamp_utc": "2026-03-02 09:00:00", "scan_event_id": "SCAN-A201"},
        {"scan_event_sequence_number": 4, "scan_location_city_state": "Denver, CO",      "scan_timestamp_utc": "2026-03-02 09:40:00", "scan_event_id": "SCAN-A202"},
        {"scan_event_sequence_number": 5, "scan_location_city_state": "Kansas City, MO", "scan_timestamp_utc": "2026-03-03 14:00:00", "scan_event_id": "SCAN-A301"},
        {"scan_event_sequence_number": 6, "scan_location_city_state": "Kansas City, MO", "scan_timestamp_utc": "2026-03-03 14:30:00", "scan_event_id": "SCAN-A302"},
        {"scan_event_sequence_number": 7, "scan_location_city_state": "Columbus, OH",    "scan_timestamp_utc": "2026-03-04 11:00:00", "scan_event_id": "SCAN-A401"},
        {"scan_event_sequence_number": 8, "scan_location_city_state": "Columbus, OH",    "scan_timestamp_utc": "2026-03-04 11:45:00", "scan_event_id": "SCAN-A402"},
    ],
}

# ─────────────────────────────────────────────────────────────────────────────
# [R3-B] SECONDARY PROBE MANIFEST — PKG-B775, 8 scan events mirroring primary.
#        Clusters: Seattle (1-2), Denver (3-4), Kansas City (5-6), Louisville (7-8).
#        THFPI Group 1 uses scan_event_1: 2026-03-01 08:15:00 (Seattle, WA) — 15 min after primary.
# ─────────────────────────────────────────────────────────────────────────────
SECONDARY_PROBE_PACKAGE_MANIFEST_RAW_DATA = {
    "package_tracking_id": "PKG-B775",
    "scan_event_history": [
        {"scan_event_sequence_number": 1, "scan_location_city_state": "Seattle, WA",     "scan_timestamp_utc": "2026-03-01 08:15:00", "scan_event_id": "SCAN-B101"},
        {"scan_event_sequence_number": 2, "scan_location_city_state": "Seattle, WA",     "scan_timestamp_utc": "2026-03-01 08:50:00", "scan_event_id": "SCAN-B102"},
        {"scan_event_sequence_number": 3, "scan_location_city_state": "Denver, CO",      "scan_timestamp_utc": "2026-03-02 09:20:00", "scan_event_id": "SCAN-B201"},
        {"scan_event_sequence_number": 4, "scan_location_city_state": "Denver, CO",      "scan_timestamp_utc": "2026-03-02 10:00:00", "scan_event_id": "SCAN-B202"},
        {"scan_event_sequence_number": 5, "scan_location_city_state": "Kansas City, MO", "scan_timestamp_utc": "2026-03-03 14:10:00", "scan_event_id": "SCAN-B301"},
        {"scan_event_sequence_number": 6, "scan_location_city_state": "Kansas City, MO", "scan_timestamp_utc": "2026-03-03 14:45:00", "scan_event_id": "SCAN-B302"},
        {"scan_event_sequence_number": 7, "scan_location_city_state": "Louisville, KY",  "scan_timestamp_utc": "2026-03-04 11:20:00", "scan_event_id": "SCAN-B401"},
        {"scan_event_sequence_number": 8, "scan_location_city_state": "Louisville, KY",  "scan_timestamp_utc": "2026-03-04 12:05:00", "scan_event_id": "SCAN-B402"},
    ],
}

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: unpack belt step data onto probe payload
# ─────────────────────────────────────────────────────────────────────────────
def unpack_belt_conveyor_step_data_onto_probe_payload(probe, step_data_list):
    for s in step_data_list:
        pos = s["step_sequence_position"]
        setattr(probe.payload, f"step{pos}_id",   s["belt_step_scan_id"])
        setattr(probe.payload, f"step{pos}_time", s["belt_step_wall_clock_offset_seconds"])
        setattr(probe.payload, f"step{pos}_loc",  s["belt_station_location_name"])

# ─────────────────────────────────────────────────────────────────────────────
# CONSTRUCT PROBES
# ─────────────────────────────────────────────────────────────────────────────
pri_probe = Arch.Data.Models.DynamicProbe(uid="ALPHA_ENTITY-SANDBOX")
sec_probe = Arch.Data.Models.DynamicProbe(uid="BETA_ENTITY-SANDBOX")

setattr(pri_probe.payload, 'calc_value', 2.0)
setattr(sec_probe.payload, 'calc_value', 3.0)

unpack_belt_conveyor_step_data_onto_probe_payload(pri_probe, PRIMARY_PROBE_BELT_CONVEYOR_STEP_DATA)
unpack_belt_conveyor_step_data_onto_probe_payload(sec_probe, SECONDARY_PROBE_BELT_CONVEYOR_STEP_DATA)

if INJECT_COMPLEX_MANIFEST_DATA_INTO_PROBE_PAYLOADS:
    # [R4] Store the raw manifest dicts directly — Orchestrator [ORC-M] converts them.
    setattr(pri_probe.payload, 'raw_manifest_data', PRIMARY_PROBE_PACKAGE_MANIFEST_RAW_DATA)
    setattr(sec_probe.payload, 'raw_manifest_data', SECONDARY_PROBE_PACKAGE_MANIFEST_RAW_DATA)

    # [R5] Also write flat manifest_data DynamicSection for the 3D hierarchy graph.
    setattr(pri_probe.payload, 'manifest_data', DynamicSection())
    setattr(sec_probe.payload, 'manifest_data', DynamicSection())
    setattr(pri_probe.payload.manifest_data, 'package_tracking_id', PRIMARY_PROBE_PACKAGE_MANIFEST_RAW_DATA["package_tracking_id"])
    setattr(sec_probe.payload.manifest_data, 'package_tracking_id', SECONDARY_PROBE_PACKAGE_MANIFEST_RAW_DATA["package_tracking_id"])

# ─────────────────────────────────────────────────────────────────────────────
# EXECUTE PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
res_jit, res_pri, res_sec = Arch.Execution.Orchestrator.run(pri_probe, sec_probe)
tel = res_pri.telemetry

# ─────────────────────────────────────────────────────────────────────────────
# OUTPUT ORDER: GRAPHS FIRST, THEN NARRATIVE LOGS
# ─────────────────────────────────────────────────────────────────────────────
display(HTML("""
<h2 style='color:#00d2ff; text-align:center; font-family:monospace;'>
  ⚡ THFPI · Dynamic Physics Simulation · Visualizing Telemetry
</h2>"""))

# 1. 3D animated phase trajectory
Arch.Observability.Visualizations.render_animated_telemetry(tel, res_pri, res_sec)

# 2. Architecture network
Arch.Observability.Visualizations.render_architecture_network(tel, res_pri, res_sec)

# 3. Hierarchical infrastructure sunburst
Arch.Observability.Visualizations.render_hierarchical_infrastructure(res_jit, res_pri, res_sec)

# 4. HF Telemetry Dashboard  ← NEW: every fire() event visualised
Arch.Observability.Visualizations.render_hf_telemetry_dashboard(tel, res_jit)

# ─────────────────────────────────────────────────────────────────────────────
# NARRATIVE LOGS
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'═'*90}")
print(f"  FACTORY PIPELINE CHRONICLE — NARRATIVE TIMELINE")
print(f"{'═'*90}\n")
Arch.Observability.StoryLog.flush_buffer()

Arch.Observability.StoryLog.dump_objects({
    "Probe Alpha (Primary)":  res_pri,
    "Probe Beta (Secondary)": res_sec,
    "JIT Core Engine":        res_jit,
})

# VarTracker full timeline summary
Arch.Observability.VarTracker.summary()

AttributeError: type object 'Models' has no attribute 'DynamicProbe'